### Sub-Common Voice evaluation

In [ ]:
!pip install -q transformers datasets evaluate jiwer
!pip install -q torchaudio librosa soundfile
!pip install -q hazm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 67.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.2 MB/s eta 0:00:00


In [ ]:
DATASET_PATH = "/content/drive/MyDrive/asr_project/asr_data/fa_test_only"

In [ ]:
import pandas as pd
import os

test_df = pd.read_csv(
    os.path.join(DATASET_PATH, "test.tsv"),
    sep="\t"
)

test_df["audio_path"] = test_df["path"].apply(
    lambda x: os.path.join(DATASET_PATH, "clips", x)
)

test_df = test_df[["audio_path", "sentence"]]

print(test_df.shape)

test_df.head()

(10519, 2)


,audio_path,sentence
0,/content/drive/MyDrive/asr_project/asr_data/fa...,از مهمونداری کنار بکشم
1,/content/drive/MyDrive/asr_project/asr_data/fa...,دعا خوان
2,/content/drive/MyDrive/asr_project/asr_data/fa...,اعتماد کرد
3,/content/drive/MyDrive/asr_project/asr_data/fa...,خب ، تو چیكار می كنی؟
4,/content/drive/MyDrive/asr_project/asr_data/fa...,آه، نه اصلاُ!


In [ ]:
import hazm
import re
import string

_normalizer = hazm.Normalizer()

chars_to_ignore = [
    ",", "?", ".", "!", "-", ";", ":", '"',
    "%", "'", "؟", "«", "»", "،", "(", ")",
    "؛", "_", "…"
]

chars_to_mapping = {
    'ك': 'ک',
    'ي': 'ی',
    'ى': 'ی',
    'أ': 'ا',
    'ؤ': 'و',
    'ئ': 'ی',
    'ة': 'ه',
    '\u200c': ' ',
    '\u200d': ' ',
}

def multiple_replace(text, mapping):
    pattern = "|".join(map(re.escape, mapping.keys()))
    return re.sub(pattern, lambda m: mapping[m.group()], text)

def normalize_persian(text):

    text = str(text).lower().strip()

    text = _normalizer.normalize(text)

    text = multiple_replace(text, chars_to_mapping)

    chars_to_ignore_regex = f"""[{"".join(chars_to_ignore)}]"""

    text = re.sub(chars_to_ignore_regex, '', text)

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

MODEL_NAME = "m3hrdadfi/wav2vec2-large-xlsr-persian-v3"

processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)

model = Wav2Vec2ForCTC.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print("Using device:", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/307 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/399 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Using device: cuda


In [ ]:
import librosa

sample_path = test_df.iloc[0]["audio_path"]

audio, sr = librosa.load(
    sample_path,
    sr=16000
)

inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt",
    padding=True
)

input_values = inputs.input_values.to(device)

with torch.no_grad():

    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)

prediction = processor.batch_decode(predicted_ids)[0]

print("Prediction:")
print(prediction)

print("\nGround Truth:")
print(test_df.iloc[0]["sentence"])

Prediction:
از مهمونداری کنار بکشم

Ground Truth:
از مهمونداری کنار بکشم


In [ ]:
benchmark_df = test_df.sample(100, random_state=42)

In [ ]:
from jiwer import wer, cer
import time
from tqdm import tqdm
import librosa

predictions = []
references = []

total_audio_duration = 0

start_time = time.time()

for _, row in tqdm(benchmark_df.iterrows(), total=len(benchmark_df)):

    audio, sr = librosa.load(
        row["audio_path"],
        sr=16000
    )

    audio_duration = len(audio) / sr
    total_audio_duration += audio_duration

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    input_values = inputs.input_values.to(device)

    with torch.no_grad():

        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    pred_text = processor.batch_decode(predicted_ids)[0]

    pred_text = normalize_persian(pred_text)
    true_text = normalize_persian(row["sentence"])

    predictions.append(pred_text)
    references.append(true_text)

total_time = time.time() - start_time

wer_score = wer(references, predictions)
cer_score = cer(references, predictions)

rtf = total_time / total_audio_duration

print(f"WER: {wer_score:.4f}")
print(f"CER: {cer_score:.4f}")
print(f"RTF: {rtf:.4f}")
print(f"Total inference time: {total_time:.2f} sec")

100%|██████████| 100/100 [00:42<00:00,  2.37it/s]

WER: 0.1763
CER: 0.0486
RTF: 0.0866
Total inference time: 42.15 sec


In [ ]:
results = pd.DataFrame({
    "reference": references,
    "prediction": predictions
})

results.to_csv(
    "/content/drive/MyDrive/asr_project/w2v_results.csv",
    index=False
)

In [ ]:
# ============================================================
# Persian ASR Benchmark - wav2vec2 XLSR Persian v3
# PSRB-inspired Benchmark Pipeline
# ============================================================

# ------------------------------------------------------------
# 1. Install Dependencies
# ------------------------------------------------------------

!pip install -q transformers
!pip install -q jiwer evaluate
!pip install -q hazm librosa soundfile torchaudio
!pip install -q pandas tqdm

# ------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------

import os
import re
import time
import random
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import hazm

from tqdm import tqdm

from jiwer import wer, cer

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC
)

# ------------------------------------------------------------
# 3. Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------------------------------------
# 4. Load Benchmark Subset
# ------------------------------------------------------------

benchmark_df = pd.read_csv(
    "/content/drive/MyDrive/asr_project/bench_sub.csv"
)

print("Benchmark samples:", len(benchmark_df))

benchmark_df = benchmark_df.dropna(
    subset=["audio_path", "sentence"]
)

benchmark_df = benchmark_df.reset_index(drop=True)

# ------------------------------------------------------------
# 5. Verify Paths
# ------------------------------------------------------------

missing = benchmark_df[
    ~benchmark_df["audio_path"].apply(os.path.exists)
]

print("Missing files:", len(missing))

# ------------------------------------------------------------
# 6. Persian Text Normalization
# ------------------------------------------------------------

normalizer = hazm.Normalizer()

PERSIAN_CHAR_MAP = {
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ؤ": "و",
    "ئ": "ی",
    "ة": "ه",
}

def normalize_persian(text):

    text = str(text)

    text = normalizer.normalize(text)

    for k, v in PERSIAN_CHAR_MAP.items():
        text = text.replace(k, v)

    # Remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # Persian spacing normalization
    text = text.replace("می ", "می")
    text = text.replace("نمی ", "نمی")

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

# ------------------------------------------------------------
# 7. Load Model
# ------------------------------------------------------------

MODEL_NAME = "m3hrdadfi/wav2vec2-large-xlsr-persian-v3"

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained(
    MODEL_NAME
)

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME
)

model = model.to(device)

print("Using device:", device)

# ------------------------------------------------------------
# 8. Test on 1 Sample First
# ------------------------------------------------------------

# benchmark_df = benchmark_df.iloc[:1].copy()

# ------------------------------------------------------------
# 9. Benchmarking
# ------------------------------------------------------------

predictions = []
references = []
sample_cers = []
audio_paths = []

total_audio_duration = 0

torch.cuda.empty_cache()

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()

start_time = time.time()

for idx, row in tqdm(
    benchmark_df.iterrows(),
    total=len(benchmark_df)
):

    try:

        audio_path = row["audio_path"]

        # ----------------------------------------------------
        # Audio duration (without decoding)
        # ----------------------------------------------------

        audio_info = sf.info(audio_path)

        audio_duration = (
            audio_info.frames /
            audio_info.samplerate
        )

        total_audio_duration += audio_duration

        # ----------------------------------------------------
        # Load audio
        # ----------------------------------------------------

        audio, sr = librosa.load(
            audio_path,
            sr=16000
        )

        # ----------------------------------------------------
        # Tokenization
        # ----------------------------------------------------

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )

        input_values = (
            inputs.input_values.to(device)
        )

        # ----------------------------------------------------
        # Inference
        # ----------------------------------------------------

        with torch.no_grad():

            logits = model(
                input_values
            ).logits

        predicted_ids = torch.argmax(
            logits,
            dim=-1
        )

        pred_text = processor.batch_decode(
            predicted_ids
        )[0]

        # ----------------------------------------------------
        # Normalize texts
        # ----------------------------------------------------

        pred_text = normalize_persian(
            pred_text
        )

        true_text = normalize_persian(
            row["sentence"]
        )

        # ----------------------------------------------------
        # Save outputs
        # ----------------------------------------------------

        predictions.append(pred_text)

        references.append(true_text)

        audio_paths.append(audio_path)

        sample_cer = cer(
            true_text,
            pred_text
        )

        sample_cers.append(sample_cer)

        # ----------------------------------------------------
        # Progress checkpoint
        # ----------------------------------------------------

        if len(predictions) % 50 == 0:

            checkpoint_df = pd.DataFrame({

                "audio_path": audio_paths,

                "reference": references,

                "prediction": predictions,

                "sample_cer": sample_cers
            })

            checkpoint_df.to_csv(
                "/content/drive/MyDrive/asr_project/w2v_checkpoint.csv",
                index=False
            )

            print(
                f"Checkpoint saved at "
                f"{len(predictions)} samples"
            )

    except Exception as e:

        print(f"\nFAILED SAMPLE: {audio_path}")

        print(e)

# ------------------------------------------------------------
# 10. Final Metrics
# ------------------------------------------------------------

total_time = time.time() - start_time

wer_score = wer(
    references,
    predictions
)

cer_score = cer(
    references,
    predictions
)

rtf = (
    total_time /
    total_audio_duration
)

avg_latency = (
    total_time /
    len(predictions)
)

# ------------------------------------------------------------
# 11. GPU Memory
# ------------------------------------------------------------

if device == "cuda":

    peak_memory = (
        torch.cuda.max_memory_allocated()
        / 1024**3
    )

else:

    peak_memory = 0

# ------------------------------------------------------------
# 12. Print Results
# ------------------------------------------------------------

print("\n================ BENCHMARK RESULTS ================\n")

print(f"Model: wav2vec2-large-xlsr-persian-v3")

print(f"Samples: {len(predictions)}")

print(f"WER: {wer_score:.4f}")

print(f"CER: {cer_score:.4f}")

print(f"RTF: {rtf:.4f}")

print(f"Average latency: {avg_latency:.4f} sec")

print(f"Total inference time: {total_time:.2f} sec")

print(f"Total audio duration: {total_audio_duration:.2f} sec")

print(f"Peak GPU memory: {peak_memory:.2f} GB")

print("\n===================================================\n")

# ------------------------------------------------------------
# 13. Save Detailed Results
# ------------------------------------------------------------

results_df = pd.DataFrame({

    "audio_path": audio_paths,

    "reference": references,

    "prediction": predictions,

    "sample_cer": sample_cers
})

SAVE_PATH = (
    "/content/drive/MyDrive/asr_project/"
    "wav2vec2_persian_v3_results.csv"
)

results_df.to_csv(
    SAVE_PATH,
    index=False
)

print("Detailed results saved to:")
print(SAVE_PATH)

# ------------------------------------------------------------
# 14. Save Benchmark Summary
# ------------------------------------------------------------

summary_df = pd.DataFrame([{

    "model": "wav2vec2-large-xlsr-persian-v3",

    "samples": len(predictions),

    "wer": wer_score,

    "cer": cer_score,

    "rtf": rtf,

    "avg_latency_sec": avg_latency,

    "gpu_memory_gb": peak_memory,

    "total_inference_time_sec": total_time
}])

summary_df.to_csv(
    "/content/drive/MyDrive/asr_project/w2v_summary.csv",
    index=False
)

print("Summary saved.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 66.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.9 MB/s eta 0:00:00
Benchmark samples: 1052
Missing files: 0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/307 [00:00<?, ?B/s]

...

Checkpoint saved at 1000 samples



100%|█████████▉| 1050/1052 [07:16<00:00,  2.30it/s]

Checkpoint saved at 1050 samples



100%|██████████| 1052/1052 [07:17<00:00,  2.40it/s]


================ BENCHMARK RESULTS ================

Model: wav2vec2-large-xlsr-persian-v3
Samples: 1052
WER: 0.1704
CER: 0.0443
RTF: 0.0835
Average latency: 0.4161 sec
Total inference time: 437.69 sec
Total audio duration: 5241.78 sec
Peak GPU memory: 1.38 GB


Detailed results saved to:
/content/drive/MyDrive/asr_project/wav2vec2_persian_v3_results.csv
Summary saved.


### FLEURS Evaluation

### w2vec

In [ ]:
from google.colab import drive
from pathlib import Path
import os

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")

BASE_DIR = Path(
    "/content/drive/MyDrive/persian_asr_llm_error_propagation"
)

DATA_DIR = BASE_DIR / "data"

ASR_COMPARE_DIR = BASE_DIR / "asr_comparison"

ASR_COMPARE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TSV_PATH = DATA_DIR / "test.tsv"

print("TSV:", TSV_PATH)
print("Exists:", TSV_PATH.exists())

Google Drive already mounted.
TSV: /content/drive/MyDrive/persian_asr_llm_error_propagation/data/test.tsv
Exists: True


In [ ]:
!pip install -q transformers accelerate soundfile librosa tqdm

In [ ]:
import os
import time
import hashlib
import unicodedata

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch

from tqdm.auto import tqdm

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
)

In [ ]:
COLUMNS = [
    "id",
    "file_name",
    "raw_transcription",
    "transcription",
    "character_transcription",
    "num_samples",
    "gender",
]

df = pd.read_csv(
    TSV_PATH,
    sep="\t",
    header=None,
    names=COLUMNS,
)

print("Rows:", len(df))
print("Unique files:", df["file_name"].nunique())
print("Unique semantic IDs:", df["id"].nunique())

display(df.head())

Rows: 871
Unique files: 871
Unique semantic IDs: 324


,id,file_name,raw_transcription,transcription,character_transcription,num_samples,gender
0,1735,7913564082410055971.wav,محققان دانشگاه پرینستون آمریكا و دانشگاه اوپسا...,محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسا...,م ح ق ق ا ن | د ا ن ش گ ا ه | پ ر ی ن س ت و ن ...,518400,MALE
1,1720,621633057978356932.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,193920,MALE
2,1720,13180061520623477685.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,205440,MALE
3,1720,16484235889057265269.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,223680,MALE
4,1955,17985233135634044314.wav,MS نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,ms نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,m s | ن و ع ی | ب ی م ا ر ی | ا س ت | ک ه | ب ...,218880,MALE


In [ ]:
wav_files = list(
    DATA_DIR.rglob("*.wav")
)

print("WAV files found:", len(wav_files))

for p in wav_files[:5]:
    print(p)

WAV files found: 871
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/7099844447178454280.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/13720237503121452158.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/7557020602592715699.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/11242877790416358060.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/5039781156427225122.wav


In [ ]:
wav_map = {}

duplicates = []

for p in wav_files:

    if p.name in wav_map:
        duplicates.append(p.name)

    wav_map[p.name] = str(p)

print("Unique WAV filenames:", len(wav_map))
print("Duplicate filenames:", len(set(duplicates)))

Unique WAV filenames: 871
Duplicate filenames: 0


In [ ]:
df["audio_path"] = (
    df["file_name"]
    .map(wav_map)
)

missing = (
    df["audio_path"]
    .isna()
    .sum()
)

print("Matched audio:", df["audio_path"].notna().sum())
print("Missing audio:", missing)

assert len(df) == 871
assert df["file_name"].nunique() == 871
assert missing == 0

print("All 871 FLEURS recordings mapped successfully.")

Matched audio: 871
Missing audio: 0
All 871 FLEURS recordings mapped successfully.


In [ ]:
audio_info = []

for audio_path in df["audio_path"]:

    info = sf.info(
        audio_path
    )

    audio_info.append({
        "file_name": Path(audio_path).name,
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "subtype": info.subtype,
        "duration": info.duration,
    })

audio_info_df = pd.DataFrame(
    audio_info
)

print("Sample rates:")
print(
    audio_info_df[
        "sample_rate"
    ].value_counts()
)

print("\nChannels:")
print(
    audio_info_df[
        "channels"
    ].value_counts()
)

print("\nWAV subtype:")
print(
    audio_info_df[
        "subtype"
    ].value_counts()
)

Sample rates:
sample_rate
16000    871
Name: count, dtype: int64

Channels:
channels
1    871
Name: count, dtype: int64

WAV subtype:
subtype
FLOAT    871
Name: count, dtype: int64


In [ ]:
SKIP = set([
    "ā", "š", "="
])

REPLACEMENTS = {
    "أ": "ا",
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ﯽ": "ی",
    "ﻮ": "و",
    "ە": "ه",
    "ۀ": "ه",
}

DISCARD = set([
    "!", '"', "#", "&", "'", "(", ")", ",", "-", ".", ":",
    ";", "؟", "،", "؛", "ـ", "…", "«", "»", "–",
    "ً", "ٌ", "َ", "ُ", "ِ", "ّ", "ْ", "ٔ"
])


def nemo_paper_normalize(text: str) -> str:

    if text is None:
        return ""

    text = str(text)

    # Unicode normalization FIRST
    text = unicodedata.normalize(
        "NFKC",
        text
    )

    # remove hashtags
    text = " ".join(
        w
        for w in text.split()
        if not w.startswith("#")
    )

    # character replacements
    for k, v in REPLACEMENTS.items():
        text = text.replace(
            k,
            v
        )

    # remove punctuation
    for tok in DISCARD:
        text = text.replace(
            tok,
            " "
        )

    # remove Arabic standalone hamza
    text = text.replace(
        "ء",
        ""
    )

    # collapse whitespace
    text = " ".join(
        text.split()
    )

    return text

In [ ]:
def _alignment_error_rate(
    reference,
    hypothesis,
):

    n = len(reference)
    m = len(hypothesis)

    dp = [
        [0] * (m + 1)
        for _ in range(n + 1)
    ]

    ops = [
        [None] * (m + 1)
        for _ in range(n + 1)
    ]

    for i in range(1, n + 1):
        dp[i][0] = i
        ops[i][0] = "D"

    for j in range(1, m + 1):
        dp[0][j] = j
        ops[0][j] = "I"

    for i in range(
        1,
        n + 1
    ):

        for j in range(
            1,
            m + 1
        ):

            if (
                reference[i - 1]
                == hypothesis[j - 1]
            ):

                dp[i][j] = (
                    dp[i - 1][j - 1]
                )

                ops[i][j] = "C"

            else:

                sub = (
                    dp[i - 1][j - 1]
                    + 1
                )

                delete = (
                    dp[i - 1][j]
                    + 1
                )

                insert = (
                    dp[i][j - 1]
                    + 1
                )

                best = min(
                    sub,
                    delete,
                    insert
                )

                dp[i][j] = best

                if best == sub:
                    ops[i][j] = "S"

                elif best == delete:
                    ops[i][j] = "D"

                else:
                    ops[i][j] = "I"

    i = n
    j = m

    substitutions = 0
    deletions = 0
    insertions = 0

    while i > 0 or j > 0:

        op = ops[i][j]

        if op == "C":

            i -= 1
            j -= 1

        elif op == "S":

            substitutions += 1

            i -= 1
            j -= 1

        elif op == "D":

            deletions += 1
            i -= 1

        elif op == "I":

            insertions += 1
            j -= 1

        else:

            break

    errors = (
        substitutions
        + deletions
        + insertions
    )

    error_rate = (
        errors / n
        if n > 0
        else float(errors > 0)
    )

    return (
        error_rate,
        substitutions,
        deletions,
        insertions,
        n,
    )

In [ ]:
def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):

    total_word_sub = 0
    total_word_del = 0
    total_word_ins = 0
    total_words = 0

    total_char_sub = 0
    total_char_del = 0
    total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(
        df[ref_col],
        df[hyp_col]
    ):

        ref = (
            ""
            if pd.isna(ref)
            else str(ref)
        )

        hyp = (
            ""
            if pd.isna(hyp)
            else str(hyp)
        )

        _, s, d, i, n = (
            _alignment_error_rate(
                ref.split(),
                hyp.split()
            )
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = (
            _alignment_error_rate(
                list(
                    ref.replace(
                        " ",
                        ""
                    )
                ),
                list(
                    hyp.replace(
                        " ",
                        ""
                    )
                )
            )
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub
        + total_word_del
        + total_word_ins
    ) / max(
        1,
        total_words
    )

    cer = (
        total_char_sub
        + total_char_del
        + total_char_ins
    ) / max(
        1,
        total_chars
    )

    return {
        "WER": wer,
        "CER": cer,

        "word_sub":
            total_word_sub,

        "word_del":
            total_word_del,

        "word_ins":
            total_word_ins,

        "char_sub":
            total_char_sub,

        "char_del":
            total_char_del,

        "char_ins":
            total_char_ins,

        "total_words":
            total_words,

        "total_chars":
            total_chars,
    }

In [ ]:
MODEL_NAME = (
    "m3hrdadfi/"
    "wav2vec2-large-xlsr-persian-v3"
)

processor = (
    Wav2Vec2Processor
    .from_pretrained(
        MODEL_NAME
    )
)

model = (
    Wav2Vec2ForCTC
    .from_pretrained(
        MODEL_NAME
    )
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(
    device
)

model.eval()

print(
    "Using device:",
    device
)

print(
    "Processor sample rate:",
    processor.feature_extractor.sampling_rate
)

print(
    "Model:",
    MODEL_NAME
)

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Using device: cuda
Processor sample rate: 16000
Model: m3hrdadfi/wav2vec2-large-xlsr-persian-v3


In [ ]:
assert (
    processor
    .feature_extractor
    .sampling_rate
    == 16000
)

In [ ]:
def transcribe_wav2vec2(
    audio_path,
    processor,
    model,
    device,
):

    audio, sr = sf.read(
        audio_path,
        dtype="float32"
    )

    # Stereo -> mono only if required
    if audio.ndim > 1:

        audio = np.mean(
            audio,
            axis=1
        )

    target_sr = (
        processor
        .feature_extractor
        .sampling_rate
    )

    # Normally not triggered for FLEURS
    if sr != target_sr:

        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=target_sr
        )

        sr = target_sr

    inputs = processor(
        audio,
        sampling_rate=sr,
        return_tensors="pt",
    )

    input_values = (
        inputs.input_values
        .to(device)
    )

    with torch.inference_mode():

        logits = model(
            input_values
        ).logits

    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )

    prediction = (
        processor
        .batch_decode(
            predicted_ids
        )[0]
    )

    return prediction.strip()

In [ ]:
test_row = df.iloc[0]

test_prediction = (
    transcribe_wav2vec2(
        test_row["audio_path"],
        processor,
        model,
        device,
    )
)

print("REFERENCE:")
print(
    test_row["transcription"]
)

print("\nWAV2VEC2:")
print(
    test_prediction
)

print("\nNORMALIZED REFERENCE:")
print(
    nemo_paper_normalize(
        test_row["transcription"]
    )
)

print("\nNORMALIZED HYPOTHESIS:")
print(
    nemo_paper_normalize(
        test_prediction
    )
)

REFERENCE:
محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسالا در سوئد گونه‌های جدید تکامل یافته‌ای را تنها در دو نسل گزارش دادند اگرچه اعتقاد بر این بود که به دلیل زاد و ولد بین یک سهره داروین بومی سهره زمینی متوسط و سهره کاکتوسی مهاجر یعنی سهره کاکتوسی بزرگ این روند خیلی بیشتر طول کشید

WAV2VEC2:
محققان دانشگاه پرینستون آمریکا و دانشگاه ابسال‌ها در سوئد گونه‌های جدید جدید تکاملیافته ای را تن‌ها در دو نسل گزارش دادند اگر چه اعتقاد بر این بود که به دلیل لذات و ولد بین یک سهره دارین بومی صهره زمینی متوسط و سهره کاکوسی مهاجر یعنی سهره کاکتوسی بزرگ این روند خیلی بیش‌تر طول کشید

NORMALIZED REFERENCE:
محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسالا در سوئد گونه‌های جدید تکامل یافته‌ای را تنها در دو نسل گزارش دادند اگرچه اعتقاد بر این بود که به دلیل زاد و ولد بین یک سهره داروین بومی سهره زمینی متوسط و سهره کاکتوسی مهاجر یعنی سهره کاکتوسی بزرگ این روند خیلی بیشتر طول کشید

NORMALIZED HYPOTHESIS:
محققان دانشگاه پرینستون آمریکا و دانشگاه ابسال‌ها در سوئد گونه‌های جدید جدید تکاملیافته ای را تن‌ها

In [ ]:
W2V_RESULTS_DIR = (
    ASR_COMPARE_DIR
    / "wav2vec2_large_xlsr_persian_v3"
)

W2V_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_PATH_W2V = (
    W2V_RESULTS_DIR
    / "wav2vec2_fleurs_checkpoint.csv"
)

FINAL_PATH_W2V = (
    W2V_RESULTS_DIR
    / "wav2vec2_fleurs_results_v1.csv"
)

SUMMARY_PATH_W2V = (
    W2V_RESULTS_DIR
    / "wav2vec2_fleurs_summary_v1.csv"
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH_W2V
)

print(
    "Final:",
    FINAL_PATH_W2V
)

print(
    "Summary:",
    SUMMARY_PATH_W2V
)

Checkpoint: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/wav2vec2_large_xlsr_persian_v3/wav2vec2_fleurs_checkpoint.csv
Final: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/wav2vec2_large_xlsr_persian_v3/wav2vec2_fleurs_results_v1.csv
Summary: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/wav2vec2_large_xlsr_persian_v3/wav2vec2_fleurs_summary_v1.csv


In [ ]:
def atomic_save_result_map(
    result_map,
    path,
):

    checkpoint_df = pd.DataFrame(
        list(
            result_map.values()
        )
    )

    if len(checkpoint_df):

        checkpoint_df = (
            checkpoint_df
            .sort_values(
                [
                    "id",
                    "file_name"
                ]
            )
            .reset_index(
                drop=True
            )
        )

    tmp_path = (
        str(path)
        + ".tmp"
    )

    checkpoint_df.to_csv(
        tmp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(
        tmp_path,
        path
    )

In [ ]:
w2v_results = {}

if CHECKPOINT_PATH_W2V.exists():

    previous_df = pd.read_csv(
        CHECKPOINT_PATH_W2V
    )

    for _, row in previous_df.iterrows():

        w2v_results[
            str(row["file_name"])
        ] = row.to_dict()

    print(
        "Checkpoint rows loaded:",
        len(w2v_results)
    )

else:

    print(
        "No existing Wav2Vec2 checkpoint."
    )

No existing Wav2Vec2 checkpoint.


In [ ]:
processed_w2v = {
    file_name
    for file_name, row
    in w2v_results.items()
    if row.get("status") == "ok"
}

print(
    "Already successful:",
    len(processed_w2v)
)

Already successful: 0


In [ ]:
for idx in tqdm(
    range(len(df)),
    desc="Wav2Vec2 FLEURS"
):

    row = df.iloc[idx]

    file_name = str(
        row["file_name"]
    )

    if file_name in processed_w2v:
        continue

    audio_path = (
        row["audio_path"]
    )

    try:

        info = sf.info(
            audio_path
        )

        duration = (
            info.duration
        )

        # CUDA operations are asynchronous,
        # so synchronize for valid timing.
        if device == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()

        pred_text = (
            transcribe_wav2vec2(
                audio_path,
                processor,
                model,
                device,
            )
        )

        if device == "cuda":
            torch.cuda.synchronize()

        inference_time = (
            time.perf_counter()
            - t0
        )

        w2v_results[
            file_name
        ] = {

            "id":
                int(row["id"]),

            "file_name":
                file_name,

            "audio_path":
                audio_path,

            "gender":
                row["gender"],

            "audio_duration":
                duration,

            "reference_raw":
                row[
                    "raw_transcription"
                ],

            # Official FLEURS reference
            # used for scoring
            "reference_normalized":
                row[
                    "transcription"
                ],

            "prediction_raw":
                pred_text,

            "inference_time":
                inference_time,

            "rtf": (
                inference_time
                / duration
                if duration > 0
                else None
            ),

            "status":
                "ok",

            "error":
                "",
        }

        processed_w2v.add(
            file_name
        )

    except Exception as e:

        w2v_results[
            file_name
        ] = {

            "id":
                int(row["id"]),

            "file_name":
                file_name,

            "audio_path":
                audio_path,

            "gender":
                row["gender"],

            "audio_duration":
                None,

            "reference_raw":
                row[
                    "raw_transcription"
                ],

            "reference_normalized":
                row[
                    "transcription"
                ],

            "prediction_raw":
                "",

            "inference_time":
                None,

            "rtf":
                None,

            "status":
                "error",

            "error":
                repr(e),
        }

        print(
            f"\nFAILED: {file_name}"
        )

        print(e)

    # Save after every recording
    atomic_save_result_map(
        w2v_results,
        CHECKPOINT_PATH_W2V
    )

Wav2Vec2 FLEURS:   0%|          | 0/871 [00:00<?, ?it/s]

In [ ]:
w2v_df = pd.read_csv(
    CHECKPOINT_PATH_W2V
)

print(
    "Checkpoint rows:",
    len(w2v_df)
)

print()

print(
    w2v_df[
        "status"
    ].value_counts(
        dropna=False
    )
)

Checkpoint rows: 871

status
ok    871
Name: count, dtype: int64


In [ ]:
w2v_sub = (
    w2v_df[
        w2v_df["status"]
        == "ok"
    ]
    .copy()
)

print(
    "Successful:",
    len(w2v_sub)
)

print(
    "Failed:",
    len(w2v_df)
    - len(w2v_sub)
)

print(
    "Unique files:",
    w2v_sub[
        "file_name"
    ].nunique()
)

Successful: 871
Failed: 0
Unique files: 871


In [ ]:
w2v_sub["hyp_norm"] = (
    w2v_sub[
        "prediction_raw"
    ]
    .apply(
        nemo_paper_normalize
    )
)

w2v_sub["ref_norm"] = (
    w2v_sub[
        "reference_normalized"
    ]
    .apply(
        nemo_paper_normalize
    )
)

In [ ]:
display(
    w2v_sub[
        [
            "reference_normalized",
            "prediction_raw",
            "ref_norm",
            "hyp_norm",
        ]
    ].head()
)

,reference_normalized,prediction_raw,ref_norm,hyp_norm
0,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,رمانتیسم اثر عزیمی از جعبگرایی فرهنگی داشت که ...,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,رمانتیسم اثر عزیمی از جعبگرایی فرهنگی داشت که ...
1,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,رمان‌تیسم آنسل عظیمی از جبلگرای فرهنگی داشت که...,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,رمان‌تیسم آنسل عظیمی از جبلگرای فرهنگی داشت که...
2,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی را برای این کاهش‌ها تعاییم نکرد و گفت ...,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی را برای این کاهش‌ها تعاییم نکرد و گفت ...
3,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رغمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رغمی برای این کاهش‌ها تعیین نکرد و گفت که ب...
4,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رغمی برای این کاهش‌ها تغییر تعیین نکرد و گف...,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رغمی برای این کاهش‌ها تغییر تعیین نکرد و گف...


In [ ]:
w2v_metrics = (
    compute_dataset_wer_cer(
        w2v_sub,
        ref_col="ref_norm",
        hyp_col="hyp_norm",
    )
)

print(
    w2v_metrics
)

{'WER': 0.2955755735828398, 'CER': 0.09038918063448673, 'word_sub': 4462, 'word_del': 467, 'word_ins': 1010, 'char_sub': 2703, 'char_del': 2043, 'char_ins': 2793, 'total_words': 20093, 'total_chars': 83406}


In [ ]:
print(
    f"Samples: {len(w2v_sub)}"
)

print(
    f"WER: {w2v_metrics['WER']:.6f}"
)

print(
    f"CER: {w2v_metrics['CER']:.6f}"
)

print(
    f"WER (%): "
    f"{w2v_metrics['WER'] * 100:.2f}%"
)

print(
    f"CER (%): "
    f"{w2v_metrics['CER'] * 100:.2f}%"
)

Samples: 871
WER: 0.295576
CER: 0.090389
WER (%): 29.56%
CER (%): 9.04%


In [ ]:
def compute_row_error_rates(
    row
):

    ref = (
        ""
        if pd.isna(
            row["ref_norm"]
        )
        else str(
            row["ref_norm"]
        )
    )

    hyp = (
        ""
        if pd.isna(
            row["hyp_norm"]
        )
        else str(
            row["hyp_norm"]
        )
    )

    wer, _, _, _, _ = (
        _alignment_error_rate(
            ref.split(),
            hyp.split()
        )
    )

    cer, _, _, _, _ = (
        _alignment_error_rate(
            list(
                ref.replace(
                    " ",
                    ""
                )
            ),
            list(
                hyp.replace(
                    " ",
                    ""
                )
            )
        )
    )

    return pd.Series({
        "wer": wer,
        "cer": cer,
    })

In [ ]:
w2v_sub[
    [
        "wer",
        "cer"
    ]
] = w2v_sub.apply(
    compute_row_error_rates,
    axis=1
)

In [ ]:
w2v_summary_df = pd.DataFrame([
    {
        "Samples":
            len(w2v_sub),

        "WER":
            w2v_metrics[
                "WER"
            ],

        "CER":
            w2v_metrics[
                "CER"
            ],
    }
])

display(
    w2v_summary_df
)

,Samples,WER,CER
0,871,0.295576,0.090389


In [ ]:
assert len(w2v_sub) == 871

assert (
    w2v_sub[
        "file_name"
    ].nunique()
    == 871
)

w2v_sub = (
    w2v_sub
    .sort_values(
        [
            "id",
            "file_name"
        ]
    )
    .reset_index(
        drop=True
    )
)

w2v_sub.to_csv(
    FINAL_PATH_W2V,
    index=False,
    encoding="utf-8-sig"
)

w2v_summary_df.to_csv(
    SUMMARY_PATH_W2V,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Final predictions:",
    FINAL_PATH_W2V
)

print(
    "Summary:",
    SUMMARY_PATH_W2V
)

Final predictions: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/wav2vec2_large_xlsr_persian_v3/wav2vec2_fleurs_results_v1.csv
Summary: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/wav2vec2_large_xlsr_persian_v3/wav2vec2_fleurs_summary_v1.csv


In [ ]:
def sha256_file(
    path
):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(
                    1024 * 1024
                ),
            b"",
        ):

            h.update(
                chunk
            )

    return h.hexdigest()

In [ ]:
print(
    "Prediction SHA-256:"
)

print(
    sha256_file(
        FINAL_PATH_W2V
    )
)

print()

print(
    "Summary SHA-256:"
)

print(
    sha256_file(
        SUMMARY_PATH_W2V
    )
)

Prediction SHA-256:
020926d6edc7b1f5f4d20cf2c4a11b731c04f6119214e47626aa56dcb12c0505

Summary SHA-256:
2eda9373b0ba091396b388eec4e341657b98c076177130b0cbc088b92dd56c29


In [ ]:
print(
    w2v_summary_df
)

print()

print(
    w2v_metrics
)

   Samples       WER       CER
0      871  0.295576  0.090389

{'WER': 0.2955755735828398, 'CER': 0.09038918063448673, 'word_sub': 4462, 'word_del': 467, 'word_ins': 1010, 'char_sub': 2703, 'char_del': 2043, 'char_ins': 2793, 'total_words': 20093, 'total_chars': 83406}


### Whisper

In [ ]:
!pip install -q -U openai-whisper
!pip install -q soundfile librosa pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 39.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import time
import hashlib
import unicodedata

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import whisper

from tqdm.auto import tqdm

In [ ]:
COLUMNS = [
    "id",
    "file_name",
    "raw_transcription",
    "transcription",
    "character_transcription",
    "num_samples",
    "gender",
]

df = pd.read_csv(
    TSV_PATH,
    sep="\t",
    header=None,
    names=COLUMNS,
)

print("Rows:", len(df))
print("Unique files:", df["file_name"].nunique())
print("Unique semantic IDs:", df["id"].nunique())

display(df.head())

Rows: 871
Unique files: 871
Unique semantic IDs: 324


,id,file_name,raw_transcription,transcription,character_transcription,num_samples,gender
0,1735,7913564082410055971.wav,محققان دانشگاه پرینستون آمریكا و دانشگاه اوپسا...,محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسا...,م ح ق ق ا ن | د ا ن ش گ ا ه | پ ر ی ن س ت و ن ...,518400,MALE
1,1720,621633057978356932.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,193920,MALE
2,1720,13180061520623477685.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,205440,MALE
3,1720,16484235889057265269.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,223680,MALE
4,1955,17985233135634044314.wav,MS نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,ms نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,m s | ن و ع ی | ب ی م ا ر ی | ا س ت | ک ه | ب ...,218880,MALE


In [ ]:
df["audio_path"] = (
    df["file_name"]
    .map(wav_map)
)

missing = (
    df["audio_path"]
    .isna()
    .sum()
)

print(
    "Matched:",
    df["audio_path"].notna().sum()
)

print(
    "Missing:",
    missing
)

assert len(df) == 871
assert df["file_name"].nunique() == 871
assert missing == 0

print(
    "All 871 recordings mapped."
)

Matched: 871
Missing: 0
All 871 recordings mapped.


In [ ]:
SKIP = set([
    "ā", "š", "="
])

REPLACEMENTS = {
    "أ": "ا",
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ﯽ": "ی",
    "ﻮ": "و",
    "ە": "ه",
    "ۀ": "ه",
}

DISCARD = set([
    "!", '"', "#", "&", "'", "(", ")", ",", "-", ".", ":",
    ";", "؟", "،", "؛", "ـ", "…", "«", "»", "–",
    "ً", "ٌ", "َ", "ُ", "ِ", "ّ", "ْ", "ٔ"
])


def nemo_paper_normalize(text: str) -> str:

    if text is None:
        return ""

    text = str(text)

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = " ".join(
        w
        for w in text.split()
        if not w.startswith("#")
    )

    for k, v in REPLACEMENTS.items():

        text = text.replace(
            k,
            v
        )

    for tok in DISCARD:

        text = text.replace(
            tok,
            " "
        )

    text = text.replace(
        "ء",
        ""
    )

    text = " ".join(
        text.split()
    )

    return text

In [ ]:
def _alignment_error_rate(
    reference,
    hypothesis,
):

    n = len(reference)
    m = len(hypothesis)

    dp = [
        [0] * (m + 1)
        for _ in range(n + 1)
    ]

    ops = [
        [None] * (m + 1)
        for _ in range(n + 1)
    ]

    for i in range(1, n + 1):

        dp[i][0] = i
        ops[i][0] = "D"

    for j in range(1, m + 1):

        dp[0][j] = j
        ops[0][j] = "I"

    for i in range(1, n + 1):

        for j in range(1, m + 1):

            if (
                reference[i - 1]
                == hypothesis[j - 1]
            ):

                dp[i][j] = (
                    dp[i - 1][j - 1]
                )

                ops[i][j] = "C"

            else:

                sub = (
                    dp[i - 1][j - 1]
                    + 1
                )

                delete = (
                    dp[i - 1][j]
                    + 1
                )

                insert = (
                    dp[i][j - 1]
                    + 1
                )

                best = min(
                    sub,
                    delete,
                    insert
                )

                dp[i][j] = best

                if best == sub:
                    ops[i][j] = "S"

                elif best == delete:
                    ops[i][j] = "D"

                else:
                    ops[i][j] = "I"

    i = n
    j = m

    substitutions = 0
    deletions = 0
    insertions = 0

    while i > 0 or j > 0:

        op = ops[i][j]

        if op == "C":

            i -= 1
            j -= 1

        elif op == "S":

            substitutions += 1
            i -= 1
            j -= 1

        elif op == "D":

            deletions += 1
            i -= 1

        elif op == "I":

            insertions += 1
            j -= 1

        else:

            break

    errors = (
        substitutions
        + deletions
        + insertions
    )

    error_rate = (
        errors / n
        if n > 0
        else float(errors > 0)
    )

    return (
        error_rate,
        substitutions,
        deletions,
        insertions,
        n,
    )

In [ ]:
def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):

    total_word_sub = 0
    total_word_del = 0
    total_word_ins = 0
    total_words = 0

    total_char_sub = 0
    total_char_del = 0
    total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(
        df[ref_col],
        df[hyp_col]
    ):

        ref = (
            ""
            if pd.isna(ref)
            else str(ref)
        )

        hyp = (
            ""
            if pd.isna(hyp)
            else str(hyp)
        )

        _, s, d, i, n = (
            _alignment_error_rate(
                ref.split(),
                hyp.split()
            )
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = (
            _alignment_error_rate(
                list(
                    ref.replace(
                        " ",
                        ""
                    )
                ),
                list(
                    hyp.replace(
                        " ",
                        ""
                    )
                )
            )
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub
        + total_word_del
        + total_word_ins
    ) / max(
        1,
        total_words
    )

    cer = (
        total_char_sub
        + total_char_del
        + total_char_ins
    ) / max(
        1,
        total_chars
    )

    return {
        "WER": wer,
        "CER": cer,

        "word_sub":
            total_word_sub,

        "word_del":
            total_word_del,

        "word_ins":
            total_word_ins,

        "char_sub":
            total_char_sub,

        "char_del":
            total_char_del,

        "char_ins":
            total_char_ins,

        "total_words":
            total_words,

        "total_chars":
            total_chars,
    }

In [ ]:
MODEL_NAME = "large-v3"

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = whisper.load_model(
    MODEL_NAME,
    device=device
)

model.eval()

print(
    "Using device:",
    device
)

print(
    "Loaded Whisper model:",
    MODEL_NAME
)

100%|██████████████████████████████████████| 2.88G/2.88G [00:22<00:00, 135MiB/s]


Using device: cuda
Loaded Whisper model: large-v3


In [ ]:
WHISPER_LANGUAGE = "fa"
WHISPER_TASK = "transcribe"
WHISPER_TEMPERATURE = 0.0

WHISPER_FP16 = (
    device == "cuda"
)

print(
    "Language:",
    WHISPER_LANGUAGE
)

print(
    "Task:",
    WHISPER_TASK
)

print(
    "Temperature:",
    WHISPER_TEMPERATURE
)

print(
    "FP16:",
    WHISPER_FP16
)

Language: fa
Task: transcribe
Temperature: 0.0
FP16: True


In [ ]:
def transcribe_whisper(
    audio_path,
    model,
):

    audio, sr = sf.read(
        audio_path,
        dtype="float32"
    )

    # Stereo -> mono only if necessary
    if audio.ndim > 1:

        audio = np.mean(
            audio,
            axis=1
        )

    # Whisper expects 16 kHz
    if sr != 16000:

        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=16000
        )

        sr = 16000

    # Make sure numpy representation
    # is clean float32
    audio = np.asarray(
        audio,
        dtype=np.float32
    )

    result = model.transcribe(
        audio,

        language=WHISPER_LANGUAGE,

        task=WHISPER_TASK,

        temperature=WHISPER_TEMPERATURE,

        fp16=WHISPER_FP16,

        verbose=False,
    )

    prediction = (
        result
        .get(
            "text",
            ""
        )
        .strip()
    )

    return prediction

In [ ]:
test_row = df.iloc[0]

test_prediction = (
    transcribe_whisper(
        test_row["audio_path"],
        model,
    )
)

print("REFERENCE:")
print(
    test_row["transcription"]
)

print("\nWHISPER:")
print(
    test_prediction
)

print("\nNORMALIZED REFERENCE:")
print(
    nemo_paper_normalize(
        test_row["transcription"]
    )
)

print("\nNORMALIZED HYPOTHESIS:")
print(
    nemo_paper_normalize(
        test_prediction
    )
)

100%|██████████| 3240/3240 [00:11<00:00, 276.65frames/s]

REFERENCE:
محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسالا در سوئد گونه‌های جدید تکامل یافته‌ای را تنها در دو نسل گزارش دادند اگرچه اعتقاد بر این بود که به دلیل زاد و ولد بین یک سهره داروین بومی سهره زمینی متوسط و سهره کاکتوسی مهاجر یعنی سهره کاکتوسی بزرگ این روند خیلی بیشتر طول کشید

WHISPER:
محققان دانشگاه پرینستون امریکا و دانشگاه اوبسالا در سوئیت گناه های جدید تکم و لیافده ای را تنها در دو نسل گزارش دادند. اگرچه اعتقاد بر این بود که به دلیل زاد و ولد بین یک سهره داروین بومی، سهره زمنی متوسط و سهره کاکتوسی محاجر یعنی سهره کاکتوسی بزرگ، این روند خیلی بیشتر طول کشید. درست دارید.

NORMALIZED REFERENCE:
محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسالا در سوئد گونه‌های جدید تکامل یافته‌ای را تنها در دو نسل گزارش دادند اگرچه اعتقاد بر این بود که به دلیل زاد و ولد بین یک سهره داروین بومی سهره زمینی متوسط و سهره کاکتوسی مهاجر یعنی سهره کاکتوسی بزرگ این روند خیلی بیشتر طول کشید

NORMALIZED HYPOTHESIS:
محققان دانشگاه پرینستون امریکا و دانشگاه اوبسالا در سوئیت گناه های جدید تکم و لیافده ای را

In [ ]:
WHISPER_RESULTS_DIR = (
    ASR_COMPARE_DIR
    / "whisper_large_v3"
)

WHISPER_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_PATH_WHISPER = (
    WHISPER_RESULTS_DIR
    / "whisper_large_v3_fleurs_checkpoint.csv"
)

FINAL_PATH_WHISPER = (
    WHISPER_RESULTS_DIR
    / "whisper_large_v3_fleurs_results_v1.csv"
)

SUMMARY_PATH_WHISPER = (
    WHISPER_RESULTS_DIR
    / "whisper_large_v3_fleurs_summary_v1.csv"
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH_WHISPER
)

print(
    "Final:",
    FINAL_PATH_WHISPER
)

print(
    "Summary:",
    SUMMARY_PATH_WHISPER
)

Checkpoint: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/whisper_large_v3/whisper_large_v3_fleurs_checkpoint.csv
Final: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/whisper_large_v3/whisper_large_v3_fleurs_results_v1.csv
Summary: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/whisper_large_v3/whisper_large_v3_fleurs_summary_v1.csv


In [ ]:
def atomic_save_result_map(
    result_map,
    path,
):

    checkpoint_df = pd.DataFrame(
        list(
            result_map.values()
        )
    )

    if len(checkpoint_df):

        checkpoint_df = (
            checkpoint_df
            .sort_values(
                [
                    "id",
                    "file_name"
                ]
            )
            .reset_index(
                drop=True
            )
        )

    tmp_path = (
        str(path)
        + ".tmp"
    )

    checkpoint_df.to_csv(
        tmp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(
        tmp_path,
        path
    )

In [ ]:
whisper_results = {}

if CHECKPOINT_PATH_WHISPER.exists():

    previous_df = pd.read_csv(
        CHECKPOINT_PATH_WHISPER
    )

    for _, row in previous_df.iterrows():

        whisper_results[
            str(row["file_name"])
        ] = row.to_dict()

    print(
        "Checkpoint rows loaded:",
        len(whisper_results)
    )

else:

    print(
        "No Whisper checkpoint found."
    )

Checkpoint rows loaded: 850


In [ ]:
processed_whisper = {
    file_name
    for file_name, row
    in whisper_results.items()
    if row.get("status") == "ok"
}

print(
    "Already successful:",
    len(processed_whisper)
)

Already successful: 850


In [ ]:
for idx in tqdm(
    range(len(df)),
    desc="Whisper large-v3 FLEURS"
):

    row = df.iloc[idx]

    file_name = str(
        row["file_name"]
    )

    if file_name in processed_whisper:
        continue

    audio_path = (
        row["audio_path"]
    )

    try:

        info = sf.info(
            audio_path
        )

        duration = (
            info.duration
        )

        if device == "cuda":
            torch.cuda.synchronize()

        t0 = time.perf_counter()

        pred_text = (
            transcribe_whisper(
                audio_path,
                model,
            )
        )

        if device == "cuda":
            torch.cuda.synchronize()

        inference_time = (
            time.perf_counter()
            - t0
        )

        whisper_results[
            file_name
        ] = {

            "id":
                int(row["id"]),

            "file_name":
                file_name,

            "audio_path":
                audio_path,

            "gender":
                row["gender"],

            "audio_duration":
                duration,

            "reference_raw":
                row[
                    "raw_transcription"
                ],

            # Official FLEURS scoring reference
            "reference_normalized":
                row[
                    "transcription"
                ],

            "prediction_raw":
                pred_text,

            "inference_time":
                inference_time,

            "rtf": (
                inference_time
                / duration
                if duration > 0
                else None
            ),

            "status":
                "ok",

            "error":
                "",
        }

        processed_whisper.add(
            file_name
        )

    except Exception as e:

        whisper_results[
            file_name
        ] = {

            "id":
                int(row["id"]),

            "file_name":
                file_name,

            "audio_path":
                audio_path,

            "gender":
                row["gender"],

            "audio_duration":
                None,

            "reference_raw":
                row[
                    "raw_transcription"
                ],

            "reference_normalized":
                row[
                    "transcription"
                ],

            "prediction_raw":
                "",

            "inference_time":
                None,

            "rtf":
                None,

            "status":
                "error",

            "error":
                repr(e),
        }

        print(
            f"\nFAILED: {file_name}"
        )

        print(e)

    # Save every recording
    atomic_save_result_map(
        whisper_results,
        CHECKPOINT_PATH_WHISPER
    )

Whisper large-v3 FLEURS:   0%|          | 0/871 [00:00<?, ?it/s]


100%|██████████| 2076/2076 [00:07<00:00, 288.37frames/s]

100%|██████████| 966/966 [00:02<00:00, 382.01frames/s]

100%|██████████| 936/936 [00:03<00:00, 287.01frames/s]

100%|██████████| 1398/1398 [00:02<00:00, 551.57frames/s]

100%|██████████| 3822/3822 [00:16<00:00, 230.79frames/s]

100%|██████████| 1638/1638 [00:03<00:00, 431.83frames/s]

100%|██████████| 1362/1362 [00:04<00:00, 324.46frames/s]

100%|██████████| 1566/1566 [00:03<00:00, 427.94frames/s]

100%|██████████| 1290/1290 [00:03<00:00, 391.39frames/s]

100%|██████████| 1644/1644 [00:03<00:00, 514.61frames/s]

100%|██████████| 1704/1704 [00:03<00:00, 541.40frames/s]

100%|██████████| 1578/1578 [00:05<00:00, 312.98frames/s]

100%|██████████| 2316/2316 [00:04<00:00, 548.37frames/s]

100%|██████████| 1956/1956 [00:04<00:00, 429.95frames/s]

100%|██████████| 1854/1854 [00:05<00:00, 314.17frames/s]

100%|██████████| 2064/2064 [00:05<00:00, 377.86frames/s]

100%|██████████| 1944/1944 [00:06<00:00, 307.82frames/s]

100%|██████████| 

In [ ]:
whisper_df = pd.read_csv(
    CHECKPOINT_PATH_WHISPER
)

print(
    "Checkpoint rows:",
    len(whisper_df)
)

print()

print(
    whisper_df[
        "status"
    ].value_counts(
        dropna=False
    )
)

Checkpoint rows: 871

status
ok    871
Name: count, dtype: int64


In [ ]:
whisper_sub = (
    whisper_df[
        whisper_df["status"]
        == "ok"
    ]
    .copy()
)

print(
    "Successful:",
    len(whisper_sub)
)

print(
    "Failed:",
    len(whisper_df)
    - len(whisper_sub)
)

print(
    "Unique files:",
    whisper_sub[
        "file_name"
    ].nunique()
)

Successful: 871
Failed: 0
Unique files: 871


In [ ]:
whisper_sub["hyp_norm"] = (
    whisper_sub[
        "prediction_raw"
    ]
    .apply(
        nemo_paper_normalize
    )
)

whisper_sub["ref_norm"] = (
    whisper_sub[
        "reference_normalized"
    ]
    .apply(
        nemo_paper_normalize
    )
)

In [ ]:
display(
    whisper_sub[
        [
            "reference_normalized",
            "prediction_raw",
            "ref_norm",
            "hyp_norm",
        ]
    ].head()
)

,reference_normalized,prediction_raw,ref_norm,hyp_norm
0,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,رومانتیزم اثر عظیمی از جعبگرایی فرهنگی داشت که...,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,رومانتیزم اثر عظیمی از جعبگرایی فرهنگی داشت که...
1,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,رومانتیسم انصار عظیمی از جبرگرای فرهنگی داشت ک...,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,رومانتیسم انصار عظیمی از جبرگرای فرهنگی داشت ک...
2,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی را برای این کاهش ها تعین نکرد و گفت که...,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی را برای این کاهش ها تعین نکرد و گفت که...
3,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی برای این کاهش ها تعییم مکرد و گفت که ب...,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی برای این کاهش ها تعییم مکرد و گفت که ب...
4,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی برای این کاهش ها تغییر تعیین نکرد و گف...,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی برای این کاهش ها تغییر تعیین نکرد و گف...


In [ ]:
whisper_metrics = (
    compute_dataset_wer_cer(
        whisper_sub,
        ref_col="ref_norm",
        hyp_col="hyp_norm",
    )
)

print(
    whisper_metrics
)

{'WER': 0.31304434380132384, 'CER': 0.09131237560846941, 'word_sub': 4252, 'word_del': 261, 'word_ins': 1777, 'char_sub': 2780, 'char_del': 3038, 'char_ins': 1798, 'total_words': 20093, 'total_chars': 83406}


In [ ]:
print(
    f"Samples: {len(whisper_sub)}"
)

print(
    f"WER: {whisper_metrics['WER']:.6f}"
)

print(
    f"CER: {whisper_metrics['CER']:.6f}"
)

print(
    f"WER (%): "
    f"{whisper_metrics['WER'] * 100:.2f}%"
)

print(
    f"CER (%): "
    f"{whisper_metrics['CER'] * 100:.2f}%"
)

Samples: 871
WER: 0.313044
CER: 0.091312
WER (%): 31.30%
CER (%): 9.13%


In [ ]:
def compute_row_error_rates(
    row
):

    ref = (
        ""
        if pd.isna(
            row["ref_norm"]
        )
        else str(
            row["ref_norm"]
        )
    )

    hyp = (
        ""
        if pd.isna(
            row["hyp_norm"]
        )
        else str(
            row["hyp_norm"]
        )
    )

    wer, _, _, _, _ = (
        _alignment_error_rate(
            ref.split(),
            hyp.split()
        )
    )

    cer, _, _, _, _ = (
        _alignment_error_rate(
            list(
                ref.replace(
                    " ",
                    ""
                )
            ),
            list(
                hyp.replace(
                    " ",
                    ""
                )
            )
        )
    )

    return pd.Series({
        "wer": wer,
        "cer": cer,
    })

In [ ]:
whisper_sub[
    [
        "wer",
        "cer"
    ]
] = whisper_sub.apply(
    compute_row_error_rates,
    axis=1
)

In [ ]:
whisper_summary_df = pd.DataFrame([
    {
        "Samples":
            len(whisper_sub),

        "WER":
            whisper_metrics[
                "WER"
            ],

        "CER":
            whisper_metrics[
                "CER"
            ],
    }
])

display(
    whisper_summary_df
)

,Samples,WER,CER
0,871,0.313044,0.091312


In [ ]:
assert len(whisper_sub) == 871

assert (
    whisper_sub[
        "file_name"
    ].nunique()
    == 871
)

whisper_sub = (
    whisper_sub
    .sort_values(
        [
            "id",
            "file_name"
        ]
    )
    .reset_index(
        drop=True
    )
)

whisper_sub.to_csv(
    FINAL_PATH_WHISPER,
    index=False,
    encoding="utf-8-sig"
)

whisper_summary_df.to_csv(
    SUMMARY_PATH_WHISPER,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Final predictions:",
    FINAL_PATH_WHISPER
)

print(
    "Summary:",
    SUMMARY_PATH_WHISPER
)

Final predictions: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/whisper_large_v3/whisper_large_v3_fleurs_results_v1.csv
Summary: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/whisper_large_v3/whisper_large_v3_fleurs_summary_v1.csv


In [ ]:
def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(
                    1024 * 1024
                ),
            b"",
        ):

            h.update(
                chunk
            )

    return h.hexdigest()

In [ ]:
print(
    "Prediction SHA-256:"
)

print(
    sha256_file(
        FINAL_PATH_WHISPER
    )
)

print()

print(
    "Summary SHA-256:"
)

print(
    sha256_file(
        SUMMARY_PATH_WHISPER
    )
)

Prediction SHA-256:
3091fddf86c4feff189d1b60214dd0e2d4f4c608699aca12693ec639ddbf229f

Summary SHA-256:
1804c2cf30565e4eb54e7ba75ea531f5fa09176ce4088bb20e4b03fc654c4947


In [ ]:
print(
    whisper_summary_df
)

print()

print(
    whisper_metrics
)

   Samples       WER       CER
0      871  0.313044  0.091312

{'WER': 0.31304434380132384, 'CER': 0.09131237560846941, 'word_sub': 4252, 'word_del': 261, 'word_ins': 1777, 'char_sub': 2780, 'char_del': 3038, 'char_ins': 1798, 'total_words': 20093, 'total_chars': 83406}


# Nemo

In [ ]:
!pip install -q soundfile librosa pandas tqdm

In [ ]:
import os
import time
import hashlib
import unicodedata

import numpy as np
import pandas as pd
import soundfile as sf
import torch

from tqdm.auto import tqdm

import nemo.collections.asr as nemo_asr

In [ ]:
COLUMNS = [
    "id",
    "file_name",
    "raw_transcription",
    "transcription",
    "character_transcription",
    "num_samples",
    "gender",
]

df = pd.read_csv(
    TSV_PATH,
    sep="\t",
    header=None,
    names=COLUMNS,
)

print("Rows:", len(df))
print("Unique files:", df["file_name"].nunique())
print("Unique semantic IDs:", df["id"].nunique())

Rows: 871
Unique files: 871
Unique semantic IDs: 324


In [ ]:
wav_files = list(
    DATA_DIR.rglob("*.wav")
)

wav_map = {
    p.name: str(p)
    for p in wav_files
}

df["audio_path"] = (
    df["file_name"]
    .map(wav_map)
)

print(
    "Matched:",
    df["audio_path"].notna().sum()
)

print(
    "Missing:",
    df["audio_path"].isna().sum()
)

assert len(df) == 871
assert df["audio_path"].notna().all()

Matched: 871
Missing: 0


In [ ]:
audio_info = []

for path in df["audio_path"]:

    info = sf.info(path)

    audio_info.append({
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "subtype": info.subtype,
    })

audio_info_df = pd.DataFrame(audio_info)

print("Sample rates:")
print(
    audio_info_df["sample_rate"]
    .value_counts()
)

print("\nChannels:")
print(
    audio_info_df["channels"]
    .value_counts()
)

print("\nSubtype:")
print(
    audio_info_df["subtype"]
    .value_counts()
)

Sample rates:
sample_rate
16000    871
Name: count, dtype: int64

Channels:
channels
1    871
Name: count, dtype: int64

Subtype:
subtype
FLOAT    871
Name: count, dtype: int64


In [ ]:
SKIP = set([
    "ā", "š", "="
])

REPLACEMENTS = {
    "أ": "ا",
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ﯽ": "ی",
    "ﻮ": "و",
    "ە": "ه",
    "ۀ": "ه",
}

DISCARD = set([
    "!", '"', "#", "&", "'", "(", ")", ",", "-", ".", ":",
    ";", "؟", "،", "؛", "ـ", "…", "«", "»", "–",
    "ً", "ٌ", "َ", "ُ", "ِ", "ّ", "ْ", "ٔ"
])


def nemo_paper_normalize(text: str) -> str:

    if text is None:
        return ""

    text = str(text)

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = " ".join(
        w
        for w in text.split()
        if not w.startswith("#")
    )

    for k, v in REPLACEMENTS.items():
        text = text.replace(k, v)

    for tok in DISCARD:
        text = text.replace(tok, " ")

    text = text.replace(
        "ء",
        ""
    )

    text = " ".join(
        text.split()
    )

    return text

In [ ]:
def _alignment_error_rate(
    reference,
    hypothesis,
):

    n = len(reference)
    m = len(hypothesis)

    dp = [
        [0] * (m + 1)
        for _ in range(n + 1)
    ]

    ops = [
        [None] * (m + 1)
        for _ in range(n + 1)
    ]

    for i in range(1, n + 1):
        dp[i][0] = i
        ops[i][0] = "D"

    for j in range(1, m + 1):
        dp[0][j] = j
        ops[0][j] = "I"

    for i in range(1, n + 1):

        for j in range(1, m + 1):

            if reference[i - 1] == hypothesis[j - 1]:

                dp[i][j] = dp[i - 1][j - 1]
                ops[i][j] = "C"

            else:

                sub = dp[i - 1][j - 1] + 1
                delete = dp[i - 1][j] + 1
                insert = dp[i][j - 1] + 1

                best = min(
                    sub,
                    delete,
                    insert
                )

                dp[i][j] = best

                if best == sub:
                    ops[i][j] = "S"

                elif best == delete:
                    ops[i][j] = "D"

                else:
                    ops[i][j] = "I"

    i = n
    j = m

    substitutions = 0
    deletions = 0
    insertions = 0

    while i > 0 or j > 0:

        op = ops[i][j]

        if op == "C":
            i -= 1
            j -= 1

        elif op == "S":
            substitutions += 1
            i -= 1
            j -= 1

        elif op == "D":
            deletions += 1
            i -= 1

        elif op == "I":
            insertions += 1
            j -= 1

        else:
            break

    errors = (
        substitutions
        + deletions
        + insertions
    )

    error_rate = (
        errors / n
        if n > 0
        else float(errors > 0)
    )

    return (
        error_rate,
        substitutions,
        deletions,
        insertions,
        n,
    )

In [ ]:
def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):

    total_word_sub = 0
    total_word_del = 0
    total_word_ins = 0
    total_words = 0

    total_char_sub = 0
    total_char_del = 0
    total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(
        df[ref_col],
        df[hyp_col]
    ):

        ref = (
            ""
            if pd.isna(ref)
            else str(ref)
        )

        hyp = (
            ""
            if pd.isna(hyp)
            else str(hyp)
        )

        _, s, d, i, n = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = _alignment_error_rate(
            list(
                ref.replace(" ", "")
            ),
            list(
                hyp.replace(" ", "")
            )
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub
        + total_word_del
        + total_word_ins
    ) / max(
        1,
        total_words
    )

    cer = (
        total_char_sub
        + total_char_del
        + total_char_ins
    ) / max(
        1,
        total_chars
    )

    return {
        "WER": wer,
        "CER": cer,
        "word_sub": total_word_sub,
        "word_del": total_word_del,
        "word_ins": total_word_ins,
        "char_sub": total_char_sub,
        "char_del": total_char_del,
        "char_ins": total_char_ins,
        "total_words": total_words,
        "total_chars": total_chars,
    }

In [ ]:
MODEL_NAME = (
    "nvidia/"
    "stt_fa_fastconformer_hybrid_large"
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = (
    nemo_asr.models
    .EncDecHybridRNNTCTCBPEModel
    .from_pretrained(
        model_name=MODEL_NAME
    )
)

model = model.to(
    device
)

model.eval()

print("Device:", device)
print("Model:", MODEL_NAME)
print("Actual class:", type(model))

[NeMo I 2026-08-25 14:43:59 mixins:194] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-08-25 14:44:00 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: dummy
    sample_rate: 16000
    batch_size: 1
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 10
    min_duration: 0.5
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    use_lhotse: true
    lhotse:
      shar_path: /data_artifacts/data/shar/train
      batch_duration: 1200
      quadratic_duration: 15
      num_buckets: 10
      num_cuts_for_bins_estimate: 10000
      buffer_size: 10000
      shuffle_buffer_size: 10000
    
[NeMo W 2026-08-25 14:44:00 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a vali

[NeMo I 2026-08-25 14:44:01 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-08-25 14:44:01 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-08-25 14:44:02 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-08-25 14:44:02 save_restore_connector:287] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--stt_fa_fastconformer_hybrid_large/snapshots/249cf5bf70dda7220a60ddeeecff2f6aad8e1784/stt_fa_fastconformer_hybrid_large.nemo.
Device: cuda
Model: nvidia/stt_fa_fastconformer_hybrid_large
Actual class: <class 'nemo.collections.asr.models.hybrid_rnnt_ctc_bpe_models.EncDecHybridRNNTCTCBPEModel'>


In [ ]:
print(
    "Current decoder:",
    getattr(
        model,
        "cur_decoder",
        "attribute not available"
    )
)

Current decoder: rnnt


In [ ]:
def extract_nemo_text(output):

    # Some NeMo versions return:
    # [Hypothesis(text="...")]
    #
    # Others return:
    # ["..."]

    if isinstance(
        output,
        tuple
    ):
        output = output[0]

    if not isinstance(
        output,
        (list, tuple)
    ):
        raise TypeError(
            f"Unexpected NeMo output type: "
            f"{type(output)}"
        )

    if len(output) == 0:
        return ""

    item = output[0]

    if hasattr(
        item,
        "text"
    ):
        return str(
            item.text
        ).strip()

    if isinstance(
        item,
        str
    ):
        return item.strip()

    raise TypeError(
        "Unexpected NeMo hypothesis type: "
        f"{type(item)}"
    )

In [ ]:
def transcribe_nemo(
    audio_path,
    model,
):

    output = model.transcribe(
        [str(audio_path)],
        batch_size=1,
        verbose=False,
    )

    return extract_nemo_text(
        output
    )

In [ ]:
model.change_decoding_strategy(
    decoder_type="rnnt"
)

print(
    "Decoder after switch:",
    getattr(
        model,
        "cur_decoder",
        "not exposed"
    )
)

[NeMo I 2026-08-25 14:45:11 hybrid_rnnt_ctc_bpe_models:440] No `decoding_cfg` passed when changing decoding strategy, using internal config
[NeMo I 2026-08-25 14:45:11 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-08-25 14:45:11 hybrid_rnnt_ctc_bpe_models:479] Changed decoding strategy of the RNNT decoder to 
    model_type: rnnt
    strategy: greedy_batch
    compute_hypothesis_token_set: false
    preserve_alignments: null
    tdt_include_token_duration: null
    confidence_cfg:
      preserve_frame_confidence: false
      preserve_token_confidence: false
      preserve_word_confidence: false
      exclude_blank: true
      aggregation: min
      tdt_include_duration: false
      method_cfg:
        name: entropy
        entropy_type: tsallis
        alpha: 0.33
        entropy_norm: exp
        temperature: DEPRECATED
    fused_batch_size: null
    compute_timestamps: null
    compute_langs: fal

In [ ]:
for idx in [
    0,
    1,
    2,
]:

    row = df.iloc[idx]

    prediction = transcribe_nemo(
        row["audio_path"],
        model,
    )

    print("=" * 100)

    print(
        "FILE:",
        row["file_name"]
    )

    print("\nREFERENCE:")
    print(
        row["transcription"]
    )

    print("\nRNNT:")
    print(
        prediction
    )

[NeMo W 2026-08-25 14:45:35 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-25 14:45:35 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
[NeMo W 2026-08-25 14:45:36 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-25 14:45:36 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is

FILE: 7913564082410055971.wav

REFERENCE:
محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسالا در سوئد گونه‌های جدید تکامل یافته‌ای را تنها در دو نسل گزارش دادند اگرچه اعتقاد بر این بود که به دلیل زاد و ولد بین یک سهره داروین بومی سهره زمینی متوسط و سهره کاکتوسی مهاجر یعنی سهره کاکتوسی بزرگ این روند خیلی بیشتر طول کشید

RNNT:
محققان دانشگاه پون در سودگهای جدید تکامل را تنها در دو نسل گزارش دادند اگر چه بریم که به دلیل لذت بین یک سله دارین متوسط و سه موهاجر این روند خیلی بیشتر طول کشید


[NeMo W 2026-08-25 14:45:38 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-25 14:45:38 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


FILE: 621633057978356932.wav

REFERENCE:
شکلات داغ در حد استانداردهای بلژیکی است آبمیوه‌ها گران‌قیمت ولی عالی می‌باشند

RNNT:
شکلات داغ در استاندار بلژیکیان قیمت ولی عالی میب
FILE: 13180061520623477685.wav

REFERENCE:
شکلات داغ در حد استانداردهای بلژیکی است آبمیوه‌ها گران‌قیمت ولی عالی می‌باشند

RNNT:
شکلات داغ در استاندارد گران ولی عالی میباشد


In [ ]:
def atomic_save_result_map(
    result_map,
    path,
):

    checkpoint_df = pd.DataFrame(
        list(
            result_map.values()
        )
    )

    if len(checkpoint_df):

        checkpoint_df = (
            checkpoint_df
            .sort_values(
                [
                    "id",
                    "file_name"
                ]
            )
            .reset_index(
                drop=True
            )
        )

    tmp_path = (
        str(path)
        + ".tmp"
    )

    checkpoint_df.to_csv(
        tmp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(
        tmp_path,
        path
    )

In [ ]:
def run_nemo_fleurs(
    decoder_type,
):

    assert decoder_type in {
        "rnnt",
        "ctc",
    }

    model.change_decoding_strategy(
        decoder_type=decoder_type
    )

    results_dir = (
        ASR_COMPARE_DIR
        / f"nemo_fastconformer_{decoder_type}"
    )

    results_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    checkpoint_path = (
        results_dir
        / f"nemo_{decoder_type}_fleurs_checkpoint.csv"
    )

    final_path = (
        results_dir
        / f"nemo_{decoder_type}_fleurs_results_v1.csv"
    )

    summary_path = (
        results_dir
        / f"nemo_{decoder_type}_fleurs_summary_v1.csv"
    )

    results = {}

    if checkpoint_path.exists():

        old = pd.read_csv(
            checkpoint_path
        )

        for _, old_row in old.iterrows():

            results[
                str(
                    old_row["file_name"]
                )
            ] = old_row.to_dict()

        print(
            f"Loaded {len(results)} "
            f"{decoder_type.upper()} checkpoint rows"
        )

    processed = {
        file_name
        for file_name, item
        in results.items()
        if item.get("status") == "ok"
    }

    print(
        "Already completed:",
        len(processed)
    )

    for idx in tqdm(
        range(len(df)),
        desc=f"NeMo {decoder_type.upper()} FLEURS"
    ):

        row = df.iloc[idx]

        file_name = str(
            row["file_name"]
        )

        if file_name in processed:
            continue

        audio_path = (
            row["audio_path"]
        )

        try:

            info = sf.info(
                audio_path
            )

            duration = (
                info.duration
            )

            if device == "cuda":
                torch.cuda.synchronize()

            t0 = time.perf_counter()

            pred_text = transcribe_nemo(
                audio_path,
                model,
            )

            if device == "cuda":
                torch.cuda.synchronize()

            inference_time = (
                time.perf_counter()
                - t0
            )

            results[file_name] = {

                "id":
                    int(row["id"]),

                "file_name":
                    file_name,

                "audio_path":
                    audio_path,

                "gender":
                    row["gender"],

                "audio_duration":
                    duration,

                "reference_raw":
                    row[
                        "raw_transcription"
                    ],

                "reference_normalized":
                    row[
                        "transcription"
                    ],

                "prediction_raw":
                    pred_text,

                "decoder_type":
                    decoder_type,

                "inference_time":
                    inference_time,

                "rtf": (
                    inference_time
                    / duration
                    if duration > 0
                    else None
                ),

                "status":
                    "ok",

                "error":
                    "",
            }

            processed.add(
                file_name
            )

        except Exception as e:

            results[file_name] = {

                "id":
                    int(row["id"]),

                "file_name":
                    file_name,

                "audio_path":
                    audio_path,

                "gender":
                    row["gender"],

                "audio_duration":
                    None,

                "reference_raw":
                    row[
                        "raw_transcription"
                    ],

                "reference_normalized":
                    row[
                        "transcription"
                    ],

                "prediction_raw":
                    "",

                "decoder_type":
                    decoder_type,

                "inference_time":
                    None,

                "rtf":
                    None,

                "status":
                    "error",

                "error":
                    repr(e),
            }

            print(
                f"\nFAILED: {file_name}"
            )

            print(e)

        atomic_save_result_map(
            results,
            checkpoint_path
        )

    result_df = pd.read_csv(
        checkpoint_path
    )

    result_sub = (
        result_df[
            result_df["status"]
            == "ok"
        ]
        .copy()
    )

    print(
        "\nSuccessful:",
        len(result_sub)
    )

    print(
        "Failed:",
        len(result_df)
        - len(result_sub)
    )

    assert len(result_sub) == 871
    assert (
        result_sub[
            "file_name"
        ].nunique()
        == 871
    )

    result_sub["hyp_norm"] = (
        result_sub[
            "prediction_raw"
        ]
        .apply(
            nemo_paper_normalize
        )
    )

    result_sub["ref_norm"] = (
        result_sub[
            "reference_normalized"
        ]
        .apply(
            nemo_paper_normalize
        )
    )

    metrics = (
        compute_dataset_wer_cer(
            result_sub,
            ref_col="ref_norm",
            hyp_col="hyp_norm",
        )
    )

    summary = pd.DataFrame([
        {
            "Decoder":
                decoder_type.upper(),

            "Samples":
                len(result_sub),

            "WER":
                metrics["WER"],

            "CER":
                metrics["CER"],
        }
    ])

    result_sub.to_csv(
        final_path,
        index=False,
        encoding="utf-8-sig"
    )

    summary.to_csv(
        summary_path,
        index=False,
        encoding="utf-8-sig"
    )

    display(
        summary
    )

    print(
        metrics
    )

    return {
        "summary": summary,
        "metrics": metrics,
        "results": result_sub,
        "final_path": final_path,
        "summary_path": summary_path,
    }

In [ ]:
rnnt_run = run_nemo_fleurs(
    "rnnt"
)

[NeMo I 2026-08-25 14:46:41 hybrid_rnnt_ctc_bpe_models:440] No `decoding_cfg` passed when changing decoding strategy, using internal config
[NeMo I 2026-08-25 14:46:41 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-08-25 14:46:41 hybrid_rnnt_ctc_bpe_models:479] Changed decoding strategy of the RNNT decoder to 
    model_type: rnnt
    strategy: greedy_batch
    compute_hypothesis_token_set: false
    preserve_alignments: null
    tdt_include_token_duration: null
    confidence_cfg:
      preserve_frame_confidence: false
      preserve_token_confidence: false
      preserve_word_confidence: false
      exclude_blank: true
      aggregation: min
      tdt_include_duration: false
      method_cfg:
        name: entropy
        entropy_type: tsallis
        alpha: 0.33
        entropy_norm: exp
        temperature: DEPRECATED
    fused_batch_size: null
    compute_timestamps: null
    compute_langs: fal

NeMo RNNT FLEURS:   0%|          | 0/871 [00:00<?, ?it/s]

[NeMo W 2026-08-25 14:46:41 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-25 14:46:41 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
[NeMo W 2026-08-25 14:46:41 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-25 14:46:41 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is


Successful: 871
Failed: 0


,Decoder,Samples,WER,CER
0,RNNT,871,0.433186,0.27769


{'WER': 0.4331856865575076, 'CER': 0.2776898544469223, 'word_sub': 4423, 'word_del': 4055, 'word_ins': 226, 'char_sub': 2440, 'char_del': 20251, 'char_ins': 470, 'total_words': 20093, 'total_chars': 83406}


In [ ]:
print(
    rnnt_run["summary"]
)

print()

print(
    rnnt_run["metrics"]
)

  Decoder  Samples       WER      CER
0    RNNT      871  0.433186  0.27769

{'WER': 0.4331856865575076, 'CER': 0.2776898544469223, 'word_sub': 4423, 'word_del': 4055, 'word_ins': 226, 'char_sub': 2440, 'char_del': 20251, 'char_ins': 470, 'total_words': 20093, 'total_chars': 83406}


In [ ]:
ctc_run = run_nemo_fleurs(
    "ctc"
)

[NeMo I 2026-08-25 14:49:57 hybrid_rnnt_ctc_bpe_models:488] No `decoding_cfg` passed when changing decoding strategy, using internal config
[NeMo I 2026-08-25 14:49:57 hybrid_rnnt_ctc_bpe_models:513] Changed decoding strategy of the CTC decoder to 
    strategy: greedy
    preserve_alignments: null
    compute_timestamps: null
    word_seperator: ' '
    segment_seperators:
    - .
    - '!'
    - '?'
    segment_gap_threshold: null
    ctc_timestamp_type: all
    batch_dim_index: 0
    greedy:
      preserve_alignments: false
      compute_timestamps: false
      preserve_frame_confidence: false
      confidence_method_cfg:
        name: entropy
        entropy_type: tsallis
        alpha: 0.33
        entropy_norm: exp
        temperature: DEPRECATED
      ngram_lm_model: null
      ngram_lm_alpha: 0.0
      boosting_tree:
        model_path: null
        key_phrases_file: null
        key_phrases_list: null
        key_phrase_items_list: null
        context_score: 1.0
        depth

NeMo CTC FLEURS:   0%|          | 0/871 [00:00<?, ?it/s]

[NeMo W 2026-08-25 14:49:57 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-25 14:49:57 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
[NeMo W 2026-08-25 14:49:57 ctc_greedy_decoding:277] CTC decoding strategy 'greedy' is slower than 'greedy_batch', which implements the same exact interface. Consider changing your strategy to 'greedy_batch' for a free performance improvement.
[NeMo W 2026-08-25 14:49:57 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-08-25 14:49:57 dataloader:533] You are using 


Successful: 871
Failed: 0


,Decoder,Samples,WER,CER
0,CTC,871,0.297218,0.105796


{'WER': 0.29721793659483403, 'CER': 0.10579574610939262, 'word_sub': 4566, 'word_del': 564, 'word_ins': 842, 'char_sub': 2706, 'char_del': 3830, 'char_ins': 2288, 'total_words': 20093, 'total_chars': 83406}


In [ ]:
print(
    ctc_run["summary"]
)

print()

print(
    ctc_run["metrics"]
)

  Decoder  Samples       WER       CER
0     CTC      871  0.297218  0.105796

{'WER': 0.29721793659483403, 'CER': 0.10579574610939262, 'word_sub': 4566, 'word_del': 564, 'word_ins': 842, 'char_sub': 2706, 'char_del': 3830, 'char_ins': 2288, 'total_words': 20093, 'total_chars': 83406}


In [ ]:
nemo_decoder_comparison = pd.concat(
    [
        rnnt_run["summary"],
        ctc_run["summary"],
    ],
    ignore_index=True
)

display(
    nemo_decoder_comparison
)

,Decoder,Samples,WER,CER
0,RNNT,871,0.433186,0.277690
1,CTC,871,0.297218,0.105796


In [ ]:
!pip install -q pandas==2.2.2
!pip install -q numpy==1.26.4
!pip install -q numba==0.60.0
!pip install -q nemo_toolkit['asr']

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 97.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.2.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 98.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3,

In [ ]:
!pip install -q jiwer
!pip install -q hazm
!pip install -q soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.2/887.2 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.4 MB/s eta 0:00:00


In [ ]:
import os
import re
import time
import random
import numpy as np
import pandas as pd
import torch
import soundfile as sf
from tqdm import tqdm
from jiwer import wer, cer
import nemo.collections.asr as nemo_asr
import hazm

In [ ]:
MODEL_NAME = "nvidia/stt_fa_fastconformer_hybrid_large"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = nemo_asr.models.ASRModel.from_pretrained(MODEL_NAME)
model = model.to(device)

print("Device:", device)

stt_fa_fastconformer_hybrid_large.nemo: reconstructing file:   0%|          |  0.00B /  459MB            

stt_fa_fastconformer_hybrid_large.nemo: downloading bytes:           |  0.00B            

[NeMo I 2026-08-25 14:34:32 mixins:194] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-08-25 14:34:34 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: dummy
    sample_rate: 16000
    batch_size: 1
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 10
    min_duration: 0.5
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: fully_randomized
    bucketing_batch_size: null
    use_lhotse: true
    lhotse:
      shar_path: /data_artifacts/data/shar/train
      batch_duration: 1200
      quadratic_duration: 15
      num_buckets: 10
      num_cuts_for_bins_estimate: 10000
      buffer_size: 10000
      shuffle_buffer_size: 10000
    
[NeMo W 2026-08-25 14:34:34 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a vali

[NeMo I 2026-08-25 14:34:35 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-08-25 14:34:35 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-08-25 14:34:35 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-08-25 14:34:36 save_restore_connector:287] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /root/.cache/huggingface/hub/models--nvidia--stt_fa_fastconformer_hybrid_large/snapshots/249cf5bf70dda7220a60ddeeecff2f6aad8e1784/stt_fa_fastconformer_hybrid_large.nemo.
Device: cuda


In [ ]:
from google.colab import drive
from pathlib import Path
import os

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")

BASE_DIR = Path(
    "/content/drive/MyDrive/persian_asr_llm_error_propagation"
)

DATA_DIR = BASE_DIR / "data"

ASR_COMPARE_DIR = BASE_DIR / "asr_comparison"

ASR_COMPARE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TSV_PATH = DATA_DIR / "test.tsv"

print("TSV:", TSV_PATH)
print("Exists:", TSV_PATH.exists())

Google Drive already mounted.
TSV: /content/drive/MyDrive/persian_asr_llm_error_propagation/data/test.tsv
Exists: True


In [ ]:
import os
import time
import hashlib
import unicodedata

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch

from tqdm.auto import tqdm

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
)

In [ ]:
COLUMNS = [
    "id",
    "file_name",
    "raw_transcription",
    "transcription",
    "character_transcription",
    "num_samples",
    "gender",
]

df = pd.read_csv(
    TSV_PATH,
    sep="\t",
    header=None,
    names=COLUMNS,
)

print("Rows:", len(df))
print("Unique files:", df["file_name"].nunique())
print("Unique semantic IDs:", df["id"].nunique())

display(df.head())

Rows: 871
Unique files: 871
Unique semantic IDs: 324


,id,file_name,raw_transcription,transcription,character_transcription,num_samples,gender
0,1735,7913564082410055971.wav,محققان دانشگاه پرینستون آمریكا و دانشگاه اوپسا...,محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسا...,م ح ق ق ا ن | د ا ن ش گ ا ه | پ ر ی ن س ت و ن ...,518400,MALE
1,1720,621633057978356932.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,193920,MALE
2,1720,13180061520623477685.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,205440,MALE
3,1720,16484235889057265269.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,223680,MALE
4,1955,17985233135634044314.wav,MS نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,ms نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,m s | ن و ع ی | ب ی م ا ر ی | ا س ت | ک ه | ب ...,218880,MALE


In [ ]:
wav_files = list(
    DATA_DIR.rglob("*.wav")
)

print("WAV files found:", len(wav_files))

for p in wav_files[:5]:
    print(p)

WAV files found: 871
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/7099844447178454280.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/13720237503121452158.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/7557020602592715699.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/11242877790416358060.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/5039781156427225122.wav


In [ ]:
wav_map = {}

duplicates = []

for p in wav_files:

    if p.name in wav_map:
        duplicates.append(p.name)

    wav_map[p.name] = str(p)

print("Unique WAV filenames:", len(wav_map))
print("Duplicate filenames:", len(set(duplicates)))

Unique WAV filenames: 871
Duplicate filenames: 0


In [ ]:
df["audio_path"] = (
    df["file_name"]
    .map(wav_map)
)

missing = (
    df["audio_path"]
    .isna()
    .sum()
)

print("Matched audio:", df["audio_path"].notna().sum())
print("Missing audio:", missing)

assert len(df) == 871
assert df["file_name"].nunique() == 871
assert missing == 0

print("All 871 FLEURS recordings mapped successfully.")

Matched audio: 871
Missing audio: 0
All 871 FLEURS recordings mapped successfully.


In [ ]:
audio_info = []

for audio_path in df["audio_path"]:

    info = sf.info(
        audio_path
    )

    audio_info.append({
        "file_name": Path(audio_path).name,
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "subtype": info.subtype,
        "duration": info.duration,
    })

audio_info_df = pd.DataFrame(
    audio_info
)

print("Sample rates:")
print(
    audio_info_df[
        "sample_rate"
    ].value_counts()
)

print("\nChannels:")
print(
    audio_info_df[
        "channels"
    ].value_counts()
)

print("\nWAV subtype:")
print(
    audio_info_df[
        "subtype"
    ].value_counts()
)

Sample rates:
sample_rate
16000    871
Name: count, dtype: int64

Channels:
channels
1    871
Name: count, dtype: int64

WAV subtype:
subtype
FLOAT    871
Name: count, dtype: int64


In [ ]:
SKIP = set([
    "ā", "š", "="
])

REPLACEMENTS = {
    "أ": "ا",
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ﯽ": "ی",
    "ﻮ": "و",
    "ە": "ه",
    "ۀ": "ه",
}

DISCARD = set([
    "!", '"', "#", "&", "'", "(", ")", ",", "-", ".", ":",
    ";", "؟", "،", "؛", "ـ", "…", "«", "»", "–",
    "ً", "ٌ", "َ", "ُ", "ِ", "ّ", "ْ", "ٔ"
])


def nemo_paper_normalize(text: str) -> str:

    if text is None:
        return ""

    text = str(text)

    # Unicode normalization FIRST
    text = unicodedata.normalize(
        "NFKC",
        text
    )

    # remove hashtags
    text = " ".join(
        w
        for w in text.split()
        if not w.startswith("#")
    )

    # character replacements
    for k, v in REPLACEMENTS.items():
        text = text.replace(
            k,
            v
        )

    # remove punctuation
    for tok in DISCARD:
        text = text.replace(
            tok,
            " "
        )

    # remove Arabic standalone hamza
    text = text.replace(
        "ء",
        ""
    )

    # collapse whitespace
    text = " ".join(
        text.split()
    )

    return text

In [ ]:
def _alignment_error_rate(
    reference,
    hypothesis,
):

    n = len(reference)
    m = len(hypothesis)

    dp = [
        [0] * (m + 1)
        for _ in range(n + 1)
    ]

    ops = [
        [None] * (m + 1)
        for _ in range(n + 1)
    ]

    for i in range(1, n + 1):
        dp[i][0] = i
        ops[i][0] = "D"

    for j in range(1, m + 1):
        dp[0][j] = j
        ops[0][j] = "I"

    for i in range(
        1,
        n + 1
    ):

        for j in range(
            1,
            m + 1
        ):

            if (
                reference[i - 1]
                == hypothesis[j - 1]
            ):

                dp[i][j] = (
                    dp[i - 1][j - 1]
                )

                ops[i][j] = "C"

            else:

                sub = (
                    dp[i - 1][j - 1]
                    + 1
                )

                delete = (
                    dp[i - 1][j]
                    + 1
                )

                insert = (
                    dp[i][j - 1]
                    + 1
                )

                best = min(
                    sub,
                    delete,
                    insert
                )

                dp[i][j] = best

                if best == sub:
                    ops[i][j] = "S"

                elif best == delete:
                    ops[i][j] = "D"

                else:
                    ops[i][j] = "I"

    i = n
    j = m

    substitutions = 0
    deletions = 0
    insertions = 0

    while i > 0 or j > 0:

        op = ops[i][j]

        if op == "C":

            i -= 1
            j -= 1

        elif op == "S":

            substitutions += 1

            i -= 1
            j -= 1

        elif op == "D":

            deletions += 1
            i -= 1

        elif op == "I":

            insertions += 1
            j -= 1

        else:

            break

    errors = (
        substitutions
        + deletions
        + insertions
    )

    error_rate = (
        errors / n
        if n > 0
        else float(errors > 0)
    )

    return (
        error_rate,
        substitutions,
        deletions,
        insertions,
        n,
    )

In [ ]:
def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):

    total_word_sub = 0
    total_word_del = 0
    total_word_ins = 0
    total_words = 0

    total_char_sub = 0
    total_char_del = 0
    total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(
        df[ref_col],
        df[hyp_col]
    ):

        ref = (
            ""
            if pd.isna(ref)
            else str(ref)
        )

        hyp = (
            ""
            if pd.isna(hyp)
            else str(hyp)
        )

        _, s, d, i, n = (
            _alignment_error_rate(
                ref.split(),
                hyp.split()
            )
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = (
            _alignment_error_rate(
                list(
                    ref.replace(
                        " ",
                        ""
                    )
                ),
                list(
                    hyp.replace(
                        " ",
                        ""
                    )
                )
            )
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub
        + total_word_del
        + total_word_ins
    ) / max(
        1,
        total_words
    )

    cer = (
        total_char_sub
        + total_char_del
        + total_char_ins
    ) / max(
        1,
        total_chars
    )

    return {
        "WER": wer,
        "CER": cer,

        "word_sub":
            total_word_sub,

        "word_del":
            total_word_del,

        "word_ins":
            total_word_ins,

        "char_sub":
            total_char_sub,

        "char_del":
            total_char_del,

        "char_ins":
            total_char_ins,

        "total_words":
            total_words,

        "total_chars":
            total_chars,
    }

In [ ]:
def transcribe_wav2vec2(
    audio_path,
    processor,
    model,
    device,
):

    audio, sr = sf.read(
        audio_path,
        dtype="float32"
    )

    # Stereo -> mono only if required
    if audio.ndim > 1:

        audio = np.mean(
            audio,
            axis=1
        )

    target_sr = (
        processor
        .feature_extractor
        .sampling_rate
    )

    # Normally not triggered for FLEURS
    if sr != target_sr:

        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=target_sr
        )

        sr = target_sr

    inputs = processor(
        audio,
        sampling_rate=sr,
        return_tensors="pt",
    )

    input_values = (
        inputs.input_values
        .to(device)
    )

    with torch.inference_mode():

        logits = model(
            input_values
        ).logits

    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )

    prediction = (
        processor
        .batch_decode(
            predicted_ids
        )[0]
    )

    return prediction.strip()

In [ ]:
test_row = df.iloc[0]

test_prediction = (
    transcribe_wav2vec2(
        test_row["audio_path"],
        processor,
        model,
        device,
    )
)

print("REFERENCE:")
print(
    test_row["transcription"]
)

print("\nWAV2VEC2:")
print(
    test_prediction
)

print("\nNORMALIZED REFERENCE:")
print(
    nemo_paper_normalize(
        test_row["transcription"]
    )
)

print("\nNORMALIZED HYPOTHESIS:")
print(
    nemo_paper_normalize(
        test_prediction
    )
)

NameError: name 'processor' is not defined

# Vosk

In [ ]:
!pip install -q vosk pandas tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 37.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
from pathlib import Path
import os

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")

BASE_DIR = Path(
    "/content/drive/MyDrive/persian_asr_llm_error_propagation"
)

DATA_DIR = BASE_DIR / "data"
ASR_COMPARE_DIR = BASE_DIR / "asr_comparison"

ASR_COMPARE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TSV_PATH = DATA_DIR / "test.tsv"

print("TSV:", TSV_PATH)
print("Exists:", TSV_PATH.exists())

Google Drive already mounted.
TSV: /content/drive/MyDrive/persian_asr_llm_error_propagation/data/test.tsv
Exists: True


In [ ]:
wav_files = list(
    DATA_DIR.rglob("*.wav")
)

print("WAV files found:", len(wav_files))

for p in wav_files[:5]:
    print(p)

WAV files found: 871
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/7099844447178454280.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/13720237503121452158.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/7557020602592715699.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/11242877790416358060.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/5039781156427225122.wav


In [ ]:
wav_map = {
    p.name: str(p)
    for p in wav_files
}

print("Unique WAV filenames:", len(wav_map))

Unique WAV filenames: 871


In [ ]:
import pandas as pd

COLUMNS = [
    "id",
    "file_name",
    "raw_transcription",
    "transcription",
    "character_transcription",
    "num_samples",
    "gender",
]

df = pd.read_csv(
    TSV_PATH,
    sep="\t",
    header=None,
    names=COLUMNS,
)

print("Rows:", len(df))
print("Unique files:", df["file_name"].nunique())
print("Unique semantic IDs:", df["id"].nunique())

display(df.head())

Rows: 871
Unique files: 871
Unique semantic IDs: 324


,id,file_name,raw_transcription,transcription,character_transcription,num_samples,gender
0,1735,7913564082410055971.wav,محققان دانشگاه پرینستون آمریكا و دانشگاه اوپسا...,محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسا...,م ح ق ق ا ن | د ا ن ش گ ا ه | پ ر ی ن س ت و ن ...,518400,MALE
1,1720,621633057978356932.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,193920,MALE
2,1720,13180061520623477685.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,205440,MALE
3,1720,16484235889057265269.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,223680,MALE
4,1955,17985233135634044314.wav,MS نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,ms نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,m s | ن و ع ی | ب ی م ا ر ی | ا س ت | ک ه | ب ...,218880,MALE


In [ ]:
import pandas as pd

COLUMNS = [
    "id",
    "file_name",
    "raw_transcription",
    "transcription",
    "character_transcription",
    "num_samples",
    "gender",
]

df = pd.read_csv(
    TSV_PATH,
    sep="\t",
    header=None,
    names=COLUMNS,
)

print("Rows:", len(df))
print("Unique files:", df["file_name"].nunique())
print("Unique semantic IDs:", df["id"].nunique())

display(df.head())

Rows: 871
Unique files: 871
Unique semantic IDs: 324


,id,file_name,raw_transcription,transcription,character_transcription,num_samples,gender
0,1735,7913564082410055971.wav,محققان دانشگاه پرینستون آمریكا و دانشگاه اوپسا...,محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسا...,م ح ق ق ا ن | د ا ن ش گ ا ه | پ ر ی ن س ت و ن ...,518400,MALE
1,1720,621633057978356932.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,193920,MALE
2,1720,13180061520623477685.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,205440,MALE
3,1720,16484235889057265269.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,223680,MALE
4,1955,17985233135634044314.wav,MS نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,ms نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,m s | ن و ع ی | ب ی م ا ر ی | ا س ت | ک ه | ب ...,218880,MALE


In [ ]:
df["audio_path"] = (
    df["file_name"]
    .map(wav_map)
)

missing = df["audio_path"].isna().sum()

print("Matched:", df["audio_path"].notna().sum())
print("Missing:", missing)

assert len(df) == 871
assert missing == 0

print("All FLEURS audio files mapped successfully.")

Matched: 871
Missing: 0
All FLEURS audio files mapped successfully.


In [ ]:
import unicodedata

SKIP = set([
    "ā", "š", "="
])

REPLACEMENTS = {
    "أ": "ا",
    "ك": "ک",
    "ي": "ی",
    "ى": "ی",
    "ﯽ": "ی",
    "ﻮ": "و",
    "ە": "ه",
    "ۀ": "ه",
}

DISCARD = set([
    "!", '"', "#", "&", "'", "(", ")", ",", "-", ".", ":",
    ";", "؟", "،", "؛", "ـ", "…", "«", "»", "–",
    "ً", "ٌ", "َ", "ُ", "ِ", "ّ", "ْ", "ٔ"
])


def nemo_paper_normalize(text: str) -> str:

    if text is None:
        return ""

    text = str(text)

    # Unicode normalization FIRST
    text = unicodedata.normalize(
        "NFKC",
        text
    )

    # remove hashtags
    text = " ".join(
        w
        for w in text.split()
        if not w.startswith("#")
    )

    # character replacements
    for k, v in REPLACEMENTS.items():
        text = text.replace(k, v)

    # remove punctuation tokens
    for tok in DISCARD:
        text = text.replace(tok, " ")

    # remove standalone Arabic hamza
    text = text.replace("ء", "")

    # collapse whitespace
    text = " ".join(
        text.split()
    )

    return text

In [ ]:
def _alignment_error_rate(
    reference,
    hypothesis,
):
    """
    Levenshtein alignment.

    Returns:
        error_rate,
        substitutions,
        deletions,
        insertions,
        reference_length
    """

    n = len(reference)
    m = len(hypothesis)

    # dp[i][j] = minimum edits required
    # to transform reference[:i] into hypothesis[:j]
    dp = [
        [0] * (m + 1)
        for _ in range(n + 1)
    ]

    ops = [
        [None] * (m + 1)
        for _ in range(n + 1)
    ]

    for i in range(1, n + 1):
        dp[i][0] = i
        ops[i][0] = "D"

    for j in range(1, m + 1):
        dp[0][j] = j
        ops[0][j] = "I"

    for i in range(1, n + 1):

        for j in range(1, m + 1):

            if reference[i - 1] == hypothesis[j - 1]:

                dp[i][j] = dp[i - 1][j - 1]
                ops[i][j] = "C"

            else:

                sub = dp[i - 1][j - 1] + 1
                delete = dp[i - 1][j] + 1
                insert = dp[i][j - 1] + 1

                best = min(
                    sub,
                    delete,
                    insert
                )

                dp[i][j] = best

                # deterministic tie breaking
                if best == sub:
                    ops[i][j] = "S"
                elif best == delete:
                    ops[i][j] = "D"
                else:
                    ops[i][j] = "I"

    # backtrack
    i = n
    j = m

    substitutions = 0
    deletions = 0
    insertions = 0

    while i > 0 or j > 0:

        op = ops[i][j]

        if op == "C":
            i -= 1
            j -= 1

        elif op == "S":
            substitutions += 1
            i -= 1
            j -= 1

        elif op == "D":
            deletions += 1
            i -= 1

        elif op == "I":
            insertions += 1
            j -= 1

        else:
            break

    errors = (
        substitutions
        + deletions
        + insertions
    )

    error_rate = (
        errors / n
        if n > 0
        else float(errors > 0)
    )

    return (
        error_rate,
        substitutions,
        deletions,
        insertions,
        n,
    )

In [ ]:
def compute_dataset_wer_cer(
    df,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
):

    total_word_sub = 0
    total_word_del = 0
    total_word_ins = 0
    total_words = 0

    total_char_sub = 0
    total_char_del = 0
    total_char_ins = 0
    total_chars = 0

    for ref, hyp in zip(
        df[ref_col],
        df[hyp_col]
    ):

        ref = (
            ""
            if pd.isna(ref)
            else str(ref)
        )

        hyp = (
            ""
            if pd.isna(hyp)
            else str(hyp)
        )

        _, s, d, i, n = _alignment_error_rate(
            ref.split(),
            hyp.split()
        )

        total_word_sub += s
        total_word_del += d
        total_word_ins += i
        total_words += n

        _, s, d, i, n = _alignment_error_rate(
            list(
                ref.replace(" ", "")
            ),
            list(
                hyp.replace(" ", "")
            )
        )

        total_char_sub += s
        total_char_del += d
        total_char_ins += i
        total_chars += n

    wer = (
        total_word_sub
        + total_word_del
        + total_word_ins
    ) / max(1, total_words)

    cer = (
        total_char_sub
        + total_char_del
        + total_char_ins
    ) / max(1, total_chars)

    return {
        "WER": wer,
        "CER": cer,

        "word_sub": total_word_sub,
        "word_del": total_word_del,
        "word_ins": total_word_ins,

        "char_sub": total_char_sub,
        "char_del": total_char_del,
        "char_ins": total_char_ins,

        "total_words": total_words,
        "total_chars": total_chars,
    }

In [ ]:
!wget -nc https://alphacephei.com/vosk/models/vosk-model-fa-0.42.zip

--2026-08-25 05:59:46--  https://alphacephei.com/vosk/models/vosk-model-fa-0.42.zip
Resolving alphacephei.com (alphacephei.com)... 188.40.21.16, 2a01:4f8:13a:279f::2
Connecting to alphacephei.com (alphacephei.com)|188.40.21.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1667089770 (1.6G) [application/zip]
Saving to: ‘vosk-model-fa-0.42.zip’

vosk-model-fa-0.42. 100%[===================>]   1.55G  19.5MB/s    in 85s     

2026-08-25 06:01:12 (18.7 MB/s) - ‘vosk-model-fa-0.42.zip’ saved [1667089770/1667089770]



In [ ]:
!unzip -q -n vosk-model-fa-0.42.zip -d /content/

In [ ]:
!ls -lh /content/vosk-model-fa-0.42

total 28K
drwxr-xr-x 2 root root 4.0K Oct 26  2024 am
drwxr-xr-x 2 root root 4.0K Sep 27  2024 conf
drwxr-xr-x 3 root root 4.0K Oct 26  2024 graph
drwxr-xr-x 2 root root 4.0K Oct 14  2024 ivector
-rw-r--r-- 1 root root  425 Oct 26  2024 README
drwxr-xr-x 2 root root 4.0K Jan 16  2021 rescore
drwxr-xr-x 2 root root 4.0K Jan 16  2021 test


In [ ]:
from vosk import Model

VOSK_MODEL_PATH = (
    "/content/vosk-model-fa-0.42"
)

model = Model(
    VOSK_MODEL_PATH
)

print(
    "Vosk model loaded successfully"
)

Vosk model loaded successfully


In [ ]:
!pip install -q soundfile

In [ ]:
import soundfile as sf
import numpy as np
from pathlib import Path

In [ ]:
for path in df["audio_path"].head(5):

    info = sf.info(path)

    print(
        Path(path).name,
        "| channels:", info.channels,
        "| rate:", info.samplerate,
        "| subtype:", info.subtype,
        "| format:", info.format,
    )

7913564082410055971.wav | channels: 1 | rate: 16000 | subtype: FLOAT | format: WAV
621633057978356932.wav | channels: 1 | rate: 16000 | subtype: FLOAT | format: WAV
13180061520623477685.wav | channels: 1 | rate: 16000 | subtype: FLOAT | format: WAV
16484235889057265269.wav | channels: 1 | rate: 16000 | subtype: FLOAT | format: WAV
17985233135634044314.wav | channels: 1 | rate: 16000 | subtype: FLOAT | format: WAV


In [ ]:
audio_formats = []

for path in df["audio_path"]:

    info = sf.info(path)

    audio_formats.append({
        "file_name": Path(path).name,
        "sample_rate": info.samplerate,
        "channels": info.channels,
        "subtype": info.subtype,
        "format": info.format,
    })

audio_formats_df = pd.DataFrame(audio_formats)

print("Sample rates:")
print(audio_formats_df["sample_rate"].value_counts())

print("\nChannels:")
print(audio_formats_df["channels"].value_counts())

print("\nSubtypes:")
print(audio_formats_df["subtype"].value_counts())

Sample rates:
sample_rate
16000    871
Name: count, dtype: int64

Channels:
channels
1    871
Name: count, dtype: int64

Subtypes:
subtype
FLOAT    871
Name: count, dtype: int64


In [ ]:
import json
import numpy as np
import soundfile as sf
import librosa

from vosk import KaldiRecognizer


def transcribe_vosk(audio_path, model):

    audio, sr = sf.read(
        audio_path,
        dtype="float32"
    )

    # Stereo -> mono, only if necessary
    if audio.ndim > 1:
        audio = np.mean(
            audio,
            axis=1
        )

    # Resample only if necessary
    if sr != 16000:

        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=16000
        )

        sr = 16000

    # Prevent overflow before PCM16 conversion
    audio = np.clip(
        audio,
        -1.0,
        1.0
    )

    audio_int16 = (
        audio * 32767
    ).astype(np.int16)

    rec = KaldiRecognizer(
        model,
        16000
    )

    parts = []

    # Feed in chunks rather than one huge buffer
    chunk_size = 4000

    for start in range(
        0,
        len(audio_int16),
        chunk_size
    ):

        chunk = audio_int16[
            start:start + chunk_size
        ]

        if rec.AcceptWaveform(
            chunk.tobytes()
        ):

            result = json.loads(
                rec.Result()
            )

            text = result.get(
                "text",
                ""
            ).strip()

            if text:
                parts.append(text)

    final_result = json.loads(
        rec.FinalResult()
    )

    final_text = final_result.get(
        "text",
        ""
    ).strip()

    if final_text:
        parts.append(final_text)

    return " ".join(parts).strip()

In [ ]:
from pathlib import Path
import os
import time
import pandas as pd
from tqdm.auto import tqdm

VOSK_RESULTS_DIR = (
    BASE_DIR
    / "asr_comparison"
    / "vosk_model_fa_0_42"
)

VOSK_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_PATH = (
    VOSK_RESULTS_DIR
    / "vosk_fleurs_checkpoint.csv"
)

FINAL_PATH = (
    VOSK_RESULTS_DIR
    / "vosk_fleurs_results_v1.csv"
)

SUMMARY_PATH = (
    VOSK_RESULTS_DIR
    / "vosk_fleurs_summary_v1.csv"
)

print("Checkpoint:", CHECKPOINT_PATH)
print("Final:", FINAL_PATH)
print("Summary:", SUMMARY_PATH)

Checkpoint: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/vosk_model_fa_0_42/vosk_fleurs_checkpoint.csv
Final: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/vosk_model_fa_0_42/vosk_fleurs_results_v1.csv
Summary: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/vosk_model_fa_0_42/vosk_fleurs_summary_v1.csv


In [ ]:
def atomic_save_csv(df, path):

    tmp_path = str(path) + ".tmp"

    df.to_csv(
        tmp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(
        tmp_path,
        path
    )

In [ ]:
if CHECKPOINT_PATH.exists():

    results_df = pd.read_csv(
        CHECKPOINT_PATH
    )

    processed = set(
        results_df.loc[
            results_df["status"] == "ok",
            "file_name"
        ].astype(str)
    )

    print(
        f"Loaded checkpoint: {len(results_df)} rows"
    )

    print(
        f"Already successful: {len(processed)}"
    )

else:

    results_df = pd.DataFrame()

    processed = set()

    print("No checkpoint found. Starting from zero.")

Loaded checkpoint: 871 rows
Already successful: 871


In [ ]:
test_row = df.iloc[0]

test_prediction = transcribe_vosk(
    test_row["audio_path"],
    model
)

print("REFERENCE:")
print(
    test_row["transcription"]
)

print("\nVOSK:")
print(
    test_prediction
)

REFERENCE:
محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسالا در سوئد گونه‌های جدید تکامل یافته‌ای را تنها در دو نسل گزارش دادند اگرچه اعتقاد بر این بود که به دلیل زاد و ولد بین یک سهره داروین بومی سهره زمینی متوسط و سهره کاکتوسی مهاجر یعنی سهره کاکتوسی بزرگ این روند خیلی بیشتر طول کشید

VOSK:
محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسالا در سوئد گونه‌های جدید جدید تکامل‌یافته‌ای را تنها در دو نسل گزارش دادند اگرچه اعتقاد بر این بود که به دلیل زاد و ولد بین یک سهره داروین بومی سهره زمینه‌ی متوسط و سهره کاکتوسی مهاجر یعنی سحر کاکتوسی بزرگ این روند خیلی بیشتر طول کشید ایست


In [ ]:
for idx in tqdm(
    range(len(df)),
    desc="Vosk FLEURS"
):

    row = df.iloc[idx]

    file_name = str(
        row["file_name"]
    )

    if file_name in processed:
        continue

    audio_path = row[
        "audio_path"
    ]

    try:

        info = sf.info(
            audio_path
        )

        duration = (
            info.duration
        )

        t0 = time.perf_counter()

        pred_text = transcribe_vosk(
            audio_path,
            model
        )

        inference_time = (
            time.perf_counter()
            - t0
        )

        new_row = {

            "id": int(row["id"]),

            "file_name": file_name,

            "audio_path": audio_path,

            "gender": row["gender"],

            "audio_duration": duration,

            # Keep original FLEURS text
            "reference_raw":
                row["raw_transcription"],

            # This is the official normalized
            # FLEURS transcript used for scoring
            "reference_normalized":
                row["transcription"],

            "prediction_raw":
                pred_text,

            "inference_time":
                inference_time,

            "rtf": (
                inference_time / duration
                if duration > 0
                else None
            ),

            "status": "ok",

            "error": "",
        }

        results_df = pd.concat(
            [
                results_df,
                pd.DataFrame([new_row])
            ],
            ignore_index=True
        )

        processed.add(
            file_name
        )

    except Exception as e:

        new_row = {

            "id": int(row["id"]),

            "file_name": file_name,

            "audio_path": audio_path,

            "gender": row["gender"],

            "audio_duration": None,

            "reference_raw":
                row["raw_transcription"],

            "reference_normalized":
                row["transcription"],

            "prediction_raw": "",

            "inference_time": None,

            "rtf": None,

            "status": "error",

            "error": repr(e),
        }

        results_df = pd.concat(
            [
                results_df,
                pd.DataFrame([new_row])
            ],
            ignore_index=True
        )

        print(
            f"\nFAILED: {file_name}"
        )

        print(e)

    # Save after every recording
    atomic_save_csv(
        results_df,
        CHECKPOINT_PATH
    )

Vosk FLEURS:   0%|          | 0/871 [00:00<?, ?it/s]

In [ ]:
print(
    "Total checkpoint rows:",
    len(results_df)
)

print()

print(
    results_df["status"]
    .value_counts(
        dropna=False
    )
)

Total checkpoint rows: 871

status
ok    871
Name: count, dtype: int64


In [ ]:
df_sub = (
    results_df[
        results_df["status"] == "ok"
    ]
    .copy()
)

print(
    "Successful:",
    len(df_sub)
)

print(
    "Failed:",
    len(results_df) - len(df_sub)
)

print(
    "Unique files:",
    df_sub["file_name"].nunique()
)

Successful: 871
Failed: 0
Unique files: 871


In [ ]:
vosk_df = pd.read_csv(
    CHECKPOINT_PATH
)

print(
    "Checkpoint rows:",
    len(vosk_df)
)

print()

print(
    vosk_df[
        "status"
    ].value_counts(
        dropna=False
    )
)

Checkpoint rows: 871

status
ok    871
Name: count, dtype: int64


In [ ]:
vosk_sub = (
    vosk_df[
        vosk_df["status"]
        == "ok"
    ]
    .copy()
)

print(
    "Successful:",
    len(vosk_sub)
)

print(
    "Failed:",
    len(vosk_df)
    - len(vosk_sub)
)

print(
    "Unique files:",
    vosk_sub[
        "file_name"
    ].nunique()
)

Successful: 871
Failed: 0
Unique files: 871


In [ ]:
vosk_sub["hyp_norm"] = (
    vosk_sub[
        "prediction_raw"
    ]
    .apply(
        nemo_paper_normalize
    )
)

vosk_sub["ref_norm"] = (
    vosk_sub[
        "reference_normalized"
    ]
    .apply(
        nemo_paper_normalize
    )
)

In [ ]:
metrics = compute_dataset_wer_cer(
    vosk_sub,
    ref_col="ref_norm",
    hyp_col="hyp_norm",
)

metrics

{'WER': 0.1674712586472901,
 'CER': 0.06280123732105604,
 'word_sub': 1872,
 'word_del': 568,
 'word_ins': 925,
 'char_sub': 1426,
 'char_del': 859,
 'char_ins': 2953,
 'total_words': 20093,
 'total_chars': 83406}

In [ ]:
print(
    f"Samples: {len(df_sub)}"
)

print(
    f"WER: {metrics['WER']:.6f}"
)

print(
    f"CER: {metrics['CER']:.6f}"
)

print(
    f"WER (%): {metrics['WER'] * 100:.2f}%"
)

print(
    f"CER (%): {metrics['CER'] * 100:.2f}%"
)

Samples: 871
WER: 0.167471
CER: 0.062801
WER (%): 16.75%
CER (%): 6.28%


In [ ]:
summary_df = pd.DataFrame([
    {
        "Samples": len(df_sub),
        "WER": metrics["WER"],
        "CER": metrics["CER"],
    }
])

display(summary_df)

,Samples,WER,CER
0,871,0.167471,0.062801


In [ ]:
summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Summary saved to:",
    SUMMARY_PATH
)

Summary saved to: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/vosk_model_fa_0_42/vosk_fleurs_summary_v1.csv


In [ ]:
assert len(df_sub) == 871
assert df_sub["file_name"].nunique() == 871

df_sub = (
    df_sub
    .sort_values(
        [
            "id",
            "file_name"
        ]
    )
    .reset_index(
        drop=True
    )
)

df_sub.to_csv(
    FINAL_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Final results saved to:",
    FINAL_PATH
)

Final results saved to: /content/drive/MyDrive/persian_asr_llm_error_propagation/asr_comparison/vosk_model_fa_0_42/vosk_fleurs_results_v1.csv


In [ ]:
import hashlib


def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()

In [ ]:
print(
    "Final prediction SHA-256:"
)

print(
    sha256_file(
        FINAL_PATH
    )
)

print()

print(
    "Summary SHA-256:"
)

print(
    sha256_file(
        SUMMARY_PATH
    )
)

Final prediction SHA-256:
d530509ec3bfdc0e08ebead28c46dcf800d1300a4aeabcec41774ef47e5e8a7a

Summary SHA-256:
b2d2bcddd8e6cf776e3ad09f4e09b9518ffead2a4ed69a7f46881abda1d90e2d


In [ ]:
print(summary_df)

print()

print(metrics)

   Samples       WER       CER
0      871  0.167471  0.062801

{'WER': 0.1674712586472901, 'CER': 0.06280123732105604, 'word_sub': 1872, 'word_del': 568, 'word_ins': 925, 'char_sub': 1426, 'char_del': 859, 'char_ins': 2953, 'total_words': 20093, 'total_chars': 83406}
